In [1]:
# ============================================================
# UCDR-1: Unified Curvature → Drift → RG Run
# Standalone, production-ready, no fitting, no fluff
# ============================================================

import math
import torch
from dataclasses import dataclass

torch.set_default_dtype(torch.float64)

# ----------------------------
# CONFIG
# ----------------------------
L = 8
d = 4
beta = 6.0
mc_dirs = 64
eps_fd = 5e-3
lanczos_iters = 30
c1 = 1.0          # curvature → drift prefactor (fixed)
device = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------
# SU(2) quaternion ops
# ----------------------------
def qmul(q, r):
    a,b,c,d = q.unbind(-1)
    e,f,g,h = r.unbind(-1)
    return torch.stack([
        a*e - b*f - c*g - d*h,
        a*f + b*e + c*h - d*g,
        a*g - b*h + c*e + d*f,
        a*h + b*g - c*f + d*e
    ], dim=-1)

def qconj(q):
    a,b,c,d = q.unbind(-1)
    return torch.stack([a,-b,-c,-d], dim=-1)

def qnorm(q):
    return q / torch.sqrt((q*q).sum(dim=-1, keepdim=True).clamp_min(1e-30))

def qrand(shape):
    return qnorm(torch.randn(*shape, 4, device=device))

# ----------------------------
# LATTICE
# ----------------------------
@dataclass
class Lattice:
    L: int
    d: int = 4

    def sites(self):
        grids = torch.meshgrid(
            *[torch.arange(self.L, device=device) for _ in range(self.d)],
            indexing="ij"
        )
        return torch.stack([g.reshape(-1) for g in grids], dim=-1)

    def lin(self, x):
        mult = torch.tensor([self.L**i for i in range(self.d)], device=device)
        return (x * mult).sum(dim=-1)

# ----------------------------
# WILSON ACTION + PLAQUETTE
# ----------------------------
def wilson_action(U, lat: Lattice, beta):
    sites = lat.sites()
    lin = lat.lin
    S = torch.zeros((), device=device)
    for mu in range(lat.d):
        for nu in range(mu+1, lat.d):
            x = sites
            x_mu = x.clone(); x_mu[:,mu]=(x_mu[:,mu]+1)%lat.L
            x_nu = x.clone(); x_nu[:,nu]=(x_nu[:,nu]+1)%lat.L
            Ux = U[lin(x)]
            plaq = qmul(
                qmul(
                    qmul(Ux[:,mu], U[lin(x_mu)][:,nu]),
                    qconj(U[lin(x_nu)][:,mu])
                ),
                qconj(Ux[:,nu])
            )
            S += (1.0 - 0.5*plaq[:,0]).sum()
    return beta * S

def V_bar(U, lat):
    sites = lat.sites()
    lin = lat.lin
    acc = 0.0
    cnt = 0
    for mu in range(lat.d):
        for nu in range(mu+1, lat.d):
            x = sites
            x_mu = x.clone(); x_mu[:,mu]=(x_mu[:,mu]+1)%lat.L
            x_nu = x.clone(); x_nu[:,nu]=(x_nu[:,nu]+1)%lat.L
            Ux = U[lin(x)]
            plaq = qmul(
                qmul(
                    qmul(Ux[:,mu], U[lin(x_mu)][:,nu]),
                    qconj(U[lin(x_nu)][:,mu])
                ),
                qconj(Ux[:,nu])
            )
            acc += (1.0 - 0.5*plaq[:,0]).mean()
            cnt += 1
    return 1.0 + acc / cnt

# ----------------------------
# TANGENT + HESSIAN (projected)
# ----------------------------
def proj_tangent(v, U):
    return v - (v*U).sum(dim=-1, keepdim=True)*U

def hvp(U, lat, beta, v):
    Ureq = U.detach().requires_grad_(True)
    S = wilson_action(Ureq, lat, beta)
    g = torch.autograd.grad(S, Ureq, create_graph=True)[0]
    vT = proj_tangent(v, Ureq)
    gv = (g*vT).sum()
    Hv = torch.autograd.grad(gv, Ureq)[0]
    return proj_tangent(Hv, Ureq).detach()

def lambda_min_phys(U, lat, beta):
    def A(x):
        return hvp(U, lat, beta, x)
    q = proj_tangent(torch.randn_like(U), U)
    q = q / torch.norm(q)
    alpha = []
    beta_l = []
    q_prev = torch.zeros_like(q)
    for _ in range(lanczos_iters):
        z = A(q)
        a = (q*z).sum()
        z = z - a*q - (beta_l[-1]*q_prev if beta_l else 0)
        b = torch.norm(z)
        alpha.append(a.item())
        beta_l.append(b.item())
        q_prev = q
        q = z / (b + 1e-30)
    T = torch.diag(torch.tensor(alpha)) + torch.diag(torch.tensor(beta_l[:-1]),1) + torch.diag(torch.tensor(beta_l[:-1]),-1)
    return torch.linalg.eigvalsh(T).min().item()

# ----------------------------
# DRIFT ESTIMATOR
# ----------------------------
def estimate_LV(U, lat, beta):
    V0 = V_bar(U, lat)
    lap = 0.0
    gip = 0.0
    for _ in range(mc_dirs):
        Xi = proj_tangent(torch.randn_like(U), U)
        Up = qnorm(U + eps_fd*Xi)
        Um = qnorm(U - eps_fd*Xi)
        Vp = V_bar(Up, lat)
        Vm = V_bar(Um, lat)
        Sp = wilson_action(Up, lat, beta)
        Sm = wilson_action(Um, lat, beta)
        lap += (Vp + Vm - 2*V0)/(eps_fd**2)
        gip += ((Sp-Sm)/(2*eps_fd))*((Vp-Vm)/(2*eps_fd))
    lap /= mc_dirs
    gip /= mc_dirs
    return V0, lap - gip

# ----------------------------
# BLOCKING (2×)
# ----------------------------
def block2x(U, lat):
    Lc = lat.L//2
    Ug = U.reshape([lat.L]*lat.d + [lat.d,4])
    Uc = torch.empty([Lc]*lat.d + [lat.d,4], device=device)
    even = [slice(0,lat.L,2)]*lat.d
    Ue = Ug[tuple(even)]
    for mu in range(lat.d):
        idx2 = [slice(1,lat.L,2) if i==mu else slice(0,lat.L,2) for i in range(lat.d)]
        Uc[...,mu,:] = qmul(Ue[...,mu,:], Ug[tuple(idx2)][...,mu,:])
    return qnorm(Uc.reshape(-1,lat.d,4)), Lattice(Lc,lat.d)

# ----------------------------
# MAIN RUN
# ----------------------------
lat = Lattice(L)
U = qrand((lat.L**lat.d, lat.d))

results = []

for i in range(12):
    lam = lambda_min_phys(U, lat, beta)
    V, LV = estimate_LV(U, lat, beta)

    Uc, latc = block2x(U, lat)
    lam_c = lambda_min_phys(Uc, latc, beta)
    Vc, LVc = estimate_LV(Uc, latc, beta)

    results.append((lam, V, LV, lam_c, Vc, LVc))

    U = qrand((lat.L**lat.d, lat.d))  # resample

# ----------------------------
# CHECK INEQUALITY
# ----------------------------
vals = []
for lam, V, LV, _, _, _ in results:
    vals.append(LV + c1*lam*V)

b = max(vals)

print("b =", b)
print("violations:", sum(v > b for v in vals), "/", len(vals))


b = tensor(-55.1680)
violations: tensor(0) / 12


In [2]:
delta = abs(lam_c - lam) / lam


In [1]:
# ============================================================
# UCDR-1 + RG STABILITY (GPU-ONLY, FULL BLOCK)
# Unified Curvature → Drift → RG
# ============================================================

import math
import torch
from dataclasses import dataclass

# ----------------------------
# GPU ONLY
# ----------------------------
assert torch.cuda.is_available(), "CUDA REQUIRED"
device = "cuda"
torch.set_default_dtype(torch.float64)

# ----------------------------
# CONFIG
# ----------------------------
L = 8
d = 4
beta = 6.0
mc_dirs = 64
eps_fd = 5e-3
lanczos_iters = 30
c1 = 1.0
n_samples = 12

# ----------------------------
# SU(2) quaternion ops
# ----------------------------
def qmul(q, r):
    a,b,c,d = q.unbind(-1)
    e,f,g,h = r.unbind(-1)
    return torch.stack([
        a*e - b*f - c*g - d*h,
        a*f + b*e + c*h - d*g,
        a*g - b*h + c*e + d*f,
        a*h + b*g - c*f + d*e
    ], dim=-1)

def qconj(q):
    a,b,c,d = q.unbind(-1)
    return torch.stack([a,-b,-c,-d], dim=-1)

def qnorm(q):
    return q / torch.sqrt((q*q).sum(dim=-1, keepdim=True).clamp_min(1e-30))

def qrand(shape):
    return qnorm(torch.randn(*shape, 4, device=device))

# ----------------------------
# LATTICE
# ----------------------------
@dataclass
class Lattice:
    L: int
    d: int = 4

    def sites(self):
        grids = torch.meshgrid(
            *[torch.arange(self.L, device=device) for _ in range(self.d)],
            indexing="ij"
        )
        return torch.stack([g.reshape(-1) for g in grids], dim=-1)

    def lin(self, x):
        mult = torch.tensor([self.L**i for i in range(self.d)], device=device)
        return (x * mult).sum(dim=-1)

# ----------------------------
# WILSON ACTION + V
# ----------------------------
def wilson_action(U, lat: Lattice, beta):
    sites = lat.sites()
    lin = lat.lin
    S = torch.zeros((), device=device)
    for mu in range(lat.d):
        for nu in range(mu+1, lat.d):
            x = sites
            x_mu = x.clone(); x_mu[:,mu]=(x_mu[:,mu]+1)%lat.L
            x_nu = x.clone(); x_nu[:,nu]=(x_nu[:,nu]+1)%lat.L
            Ux = U[lin(x)]
            plaq = qmul(
                qmul(
                    qmul(Ux[:,mu], U[lin(x_mu)][:,nu]),
                    qconj(U[lin(x_nu)][:,mu])
                ),
                qconj(Ux[:,nu])
            )
            S += (1.0 - 0.5*plaq[:,0]).sum()
    return beta * S

def V_bar(U, lat):
    sites = lat.sites()
    lin = lat.lin
    acc = 0.0
    cnt = 0
    for mu in range(lat.d):
        for nu in range(mu+1, lat.d):
            x = sites
            x_mu = x.clone(); x_mu[:,mu]=(x_mu[:,mu]+1)%lat.L
            x_nu = x.clone(); x_nu[:,nu]=(x_nu[:,nu]+1)%lat.L
            Ux = U[lin(x)]
            plaq = qmul(
                qmul(
                    qmul(Ux[:,mu], U[lin(x_mu)][:,nu]),
                    qconj(U[lin(x_nu)][:,mu])
                ),
                qconj(Ux[:,nu])
            )
            acc += (1.0 - 0.5*plaq[:,0]).mean()
            cnt += 1
    return 1.0 + acc / cnt

# ----------------------------
# TANGENT + HESSIAN
# ----------------------------
def proj_tangent(v, U):
    return v - (v*U).sum(dim=-1, keepdim=True)*U

def hvp(U, lat, beta, v):
    Ureq = U.detach().requires_grad_(True)
    S = wilson_action(Ureq, lat, beta)
    g = torch.autograd.grad(S, Ureq, create_graph=True)[0]
    vT = proj_tangent(v, Ureq)
    gv = (g*vT).sum()
    Hv = torch.autograd.grad(gv, Ureq)[0]
    return proj_tangent(Hv, Ureq).detach()

def lambda_min_phys(U, lat, beta):
    q = proj_tangent(torch.randn_like(U), U)
    q = q / torch.norm(q)
    q_prev = torch.zeros_like(q)
    alphas, betas = [], []
    for _ in range(lanczos_iters):
        z = hvp(U, lat, beta, q)
        a = (q*z).sum()
        z = z - a*q - (betas[-1]*q_prev if betas else 0)
        b = torch.norm(z)
        alphas.append(a.item())
        betas.append(b.item())
        q_prev = q
        q = z / (b + 1e-30)
    T = torch.diag(torch.tensor(alphas, device=device))
    T += torch.diag(torch.tensor(betas[:-1], device=device), 1)
    T += torch.diag(torch.tensor(betas[:-1], device=device), -1)
    return torch.linalg.eigvalsh(T).min().item()

# ----------------------------
# DRIFT ESTIMATOR
# ----------------------------
def estimate_LV(U, lat, beta):
    V0 = V_bar(U, lat)
    lap = 0.0
    gip = 0.0
    for _ in range(mc_dirs):
        Xi = proj_tangent(torch.randn_like(U), U)
        Up = qnorm(U + eps_fd*Xi)
        Um = qnorm(U - eps_fd*Xi)
        Vp = V_bar(Up, lat)
        Vm = V_bar(Um, lat)
        Sp = wilson_action(Up, lat, beta)
        Sm = wilson_action(Um, lat, beta)
        lap += (Vp + Vm - 2*V0)/(eps_fd**2)
        gip += ((Sp-Sm)/(2*eps_fd))*((Vp-Vm)/(2*eps_fd))
    lap /= mc_dirs
    gip /= mc_dirs
    return V0, lap - gip

# ----------------------------
# BLOCKING (2×)
# ----------------------------
def block2x(U, lat):
    Lc = lat.L//2
    Ug = U.reshape([lat.L]*lat.d + [lat.d,4])
    Uc = torch.empty([Lc]*lat.d + [lat.d,4], device=device)
    even = [slice(0,lat.L,2)]*lat.d
    Ue = Ug[tuple(even)]
    for mu in range(lat.d):
        idx2 = [slice(1,lat.L,2) if i==mu else slice(0,lat.L,2) for i in range(lat.d)]
        Uc[...,mu,:] = qmul(Ue[...,mu,:], Ug[tuple(idx2)][...,mu,:])
    return qnorm(Uc.reshape(-1,lat.d,4)), Lattice(Lc,lat.d)

# ----------------------------
# MAIN RUN
# ----------------------------
lat = Lattice(L)
U = qrand((lat.L**lat.d, lat.d))

vals = []
rg_ratio = []

for _ in range(n_samples):
    lam = lambda_min_phys(U, lat, beta)
    V, LV = estimate_LV(U, lat, beta)

    Uc, latc = block2x(U, lat)
    lam_c = lambda_min_phys(Uc, latc, beta)

    vals.append(LV + c1*lam*V)
    rg_ratio.append(lam_c / lam)

    U = qrand((lat.L**lat.d, lat.d))

vals = torch.tensor(vals, device=device)
rg_ratio = torch.tensor(rg_ratio, device=device)

b = vals.max()
viol = (vals > b).sum().item()

print("b =", b)
print("violations =", viol, "/", n_samples)
print("RG λ ratio stats:")
print("  mean =", rg_ratio.mean().item())
print("  min  =", rg_ratio.min().item())
print("  max  =", rg_ratio.max().item())


b = tensor(-54.7605, device='cuda:0')
violations = 0 / 12
RG λ ratio stats:
  mean = 0.985858240826053
  min  = 0.9368605570837762
  max  = 1.0132304304487858


In [2]:
# ============================================================
# UNIFIED GPU RUN (A100-ready): 4D 1-form "Maxwell + quartic" Langevin
#
# Unifies:
#  (1) FFT-exact curlcurl (d1^* d1) machinery + low-k mass fit tool
#  (2) drift decomposition, but DONE EXACTLY for V=S (no finite-diff noise):
#        L S = Tr(Hess S) - ||∇S||^2
#      with Tr(Hess S) = Tr(M) + 3*lam4*||A||^2
#      and M = m2 I + alpha * curlcurl
#  (3) coarse-graining diagnostic via naive 2x blocking (fine vs blocked drift fits)
#
# Outputs:
#   - NPZ with arrays (S, LV, grad2, traceH, norm2) for fine and blocked
#   - robust one-sided drift fit: LV <= -lam*S + b  (quantile-safe)
#   - low-k inverse propagator fit: 1/G(k) ~ m_eff^2 + Z p_hat^2 (fine and blocked)
#
# This is NOT Yang–Mills. It is a gauge-adjacent 1-form interacting sandbox.
# ============================================================

import os
import math
import time
import argparse
import numpy as np

import torch
import torch.fft as fft


# -----------------------------
# small utils
# -----------------------------
def set_seed(seed: int):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--L", type=int, default=32)
    p.add_argument("--d", type=int, default=4)
    p.add_argument("--batch", type=int, default=8)

    p.add_argument("--m2", type=float, default=0.30)
    p.add_argument("--alpha", type=float, default=1.0)
    p.add_argument("--lam4", type=float, default=0.50)

    p.add_argument("--dt", type=float, default=5e-4)
    p.add_argument("--steps", type=int, default=6000)
    p.add_argument("--burnin", type=int, default=1500)
    p.add_argument("--thin", type=int, default=20)

    p.add_argument("--init_sigma", type=float, default=0.20)
    p.add_argument("--dtype", type=str, default="float32", choices=["float32", "float64"])
    p.add_argument("--device", type=str, default="cuda")

    p.add_argument("--lowk", type=int, default=256, help="number of smallest nonzero p^2 modes for mass fit")
    p.add_argument("--qfit", type=float, default=0.995, help="quantile for one-sided drift fit")
    p.add_argument("--lam_max", type=float, default=5.0, help="max lambda grid for drift fit")
    p.add_argument("--lam_grid", type=int, default=301, help="lambda grid points for drift fit")

    p.add_argument("--out", type=str, default="unified_maxwell_phi4_drift_block.npz")
    p.add_argument("--seed", type=int, default=0)

    # notebook-safe
    args, _ = p.parse_known_args()
    return args


# -----------------------------
# FFT lattice momentum
# -----------------------------
def make_phat_1d(L: int, device: str, dtype: torch.dtype):
    # freq = n/L with signed wrap; p_hat = 2 sin(pi * freq)
    f = torch.fft.fftfreq(L, d=1.0, device=device, dtype=dtype)  # shape [L]
    return 2.0 * torch.sin(math.pi * f)


def make_pgrids(ph: torch.Tensor):
    # ph: [L]
    L = ph.numel()
    p0 = ph.view(L, 1, 1, 1)
    p1 = ph.view(1, L, 1, 1)
    p2 = ph.view(1, 1, L, 1)
    p3 = ph.view(1, 1, 1, L)
    return p0, p1, p2, p3


def project_perp_and_curlcurl(A: torch.Tensor, p0, p1, p2, p3, eps=1e-12):
    """
    A: [B,d,L,L,L,L] real
    returns:
      curlcurlA: [B,d,L,L,L,L] real
      Aperp_hat: [B,d,L,L,L,L] complex  (transverse part in Fourier)
      p2grid: [L,L,L,L] real
    """
    spatial = (-4, -3, -2, -1)
    Ahat = fft.fftn(A, dim=spatial, norm="ortho")  # complex

    # p2 grid
    p2grid = (p0 * p0 + p1 * p1 + p2 * p2 + p3 * p3)  # [L,L,L,L]

    # dot = p · Ahat  (broadcast pμ over batch)
    dot = (p0 * Ahat[:, 0] + p1 * Ahat[:, 1] + p2 * Ahat[:, 2] + p3 * Ahat[:, 3])  # [B,L,L,L,L] complex

    inv = torch.zeros_like(p2grid)
    mask = p2grid > 0
    inv[mask] = 1.0 / (p2grid[mask] + eps)  # [L,L,L,L] real

    # factor = (p·A)/p2
    factor = dot * inv  # broadcast inv to batch; complex

    # Aperp = A - p * factor
    Aperp0 = Ahat[:, 0] - p0 * factor
    Aperp1 = Ahat[:, 1] - p1 * factor
    Aperp2 = Ahat[:, 2] - p2 * factor
    Aperp3 = Ahat[:, 3] - p3 * factor
    Aperp_hat = torch.stack([Aperp0, Aperp1, Aperp2, Aperp3], dim=1)  # [B,d,L,L,L,L] complex

    # curlcurl(A) = p2 * Aperp in Fourier
    curlcurl_hat = Aperp_hat * p2grid  # broadcast p2grid
    curlcurlA = fft.ifftn(curlcurl_hat, dim=spatial, norm="ortho").real  # real

    return curlcurlA, Aperp_hat, p2grid


def energy_S(A, curlcurlA, m2, alpha, lam4):
    # S = 1/2 m2 ||A||^2 + 1/2 alpha <A, curlcurl A> + 1/4 lam4 ||A||_4^4
    # all sums over (mu,x); returns per-sample S: [B]
    mass = 0.5 * m2 * (A * A).flatten(1).sum(-1)
    curl = 0.5 * alpha * (A * curlcurlA).flatten(1).sum(-1)
    quart = 0.25 * lam4 * (A ** 4).flatten(1).sum(-1)
    return mass + curl + quart


def block2_avg(A):
    # A: [B,d,L,L,L,L] with L even
    B, d, L, _, _, _ = A.shape
    assert L % 2 == 0
    L2 = L // 2
    return A.reshape(B, d, L2, 2, L2, 2, L2, 2, L2, 2).mean(dim=(3, 5, 7, 9))


def one_sided_drift_fit(S_np, LV_np, lam_max=5.0, lam_grid=301, q=0.995):
    """
    robust one-sided fit over lambda grid:
      b(lam) = quantile_q( LV + lam*S )
      violations ~ (1-q)
    choose lam* maximizing score = lam / (b+eps)
    returns dict with lam*, b*, score*, and grid arrays.
    """
    S = np.asarray(S_np, dtype=np.float64)
    LV = np.asarray(LV_np, dtype=np.float64)
    lam_grid_vals = np.linspace(0.0, float(lam_max), int(lam_grid))

    b_vals = np.empty_like(lam_grid_vals)
    viol_vals = np.empty_like(lam_grid_vals)
    score_vals = np.empty_like(lam_grid_vals)

    eps = 1e-12
    for i, lam in enumerate(lam_grid_vals):
        z = LV + lam * S
        b = np.quantile(z, q)
        b_vals[i] = b
        viol_vals[i] = np.mean(LV > (-lam * S + b))
        score_vals[i] = lam / (b + eps)

    i_star = int(np.argmax(score_vals))
    return dict(
        lam_star=float(lam_grid_vals[i_star]),
        b_star=float(b_vals[i_star]),
        score_star=float(score_vals[i_star]),
        lam_grid=lam_grid_vals,
        b_grid=b_vals,
        viol_grid=viol_vals,
        score_grid=score_vals,
    )


def lowk_mass_fit(power_k, p2_flat, lowk_idx):
    """
    Fit 1/G(k) = m_eff^2 + Z p2  using selected low-k modes
    power_k: [K] mean |A_perp(k)|^2 over (samples,batch,mu)
    p2_flat: [N] flattened p2 grid
    lowk_idx: [K] indices into flattened grid
    """
    G = np.asarray(power_k, dtype=np.float64)
    p2 = np.asarray(p2_flat[lowk_idx], dtype=np.float64)

    # guard
    good = (G > 0) & np.isfinite(G) & np.isfinite(p2)
    G = G[good]
    p2 = p2[good]
    if G.size < 8:
        return dict(m_eff2=np.nan, Z=np.nan, n=int(G.size))

    y = 1.0 / G
    x = p2
    # linear least squares
    A = np.stack([np.ones_like(x), x], axis=1)
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    m_eff2 = float(coef[0])
    Z = float(coef[1])
    return dict(m_eff2=m_eff2, Z=Z, n=int(G.size))


def main():
    args = parse_args()

    device = args.device
    if device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA requested but not available")

    dtype = torch.float32 if args.dtype == "float32" else torch.float64
    set_seed(args.seed)

    L = args.L
    d = args.d
    B = args.batch
    assert d == 4, "this script assumes d=4 for the FFT momentum grids"

    # init field
    A = (args.init_sigma * torch.randn((B, d, L, L, L, L), device=device, dtype=dtype))

    # momentum grids fine
    ph = make_phat_1d(L, device=device, dtype=dtype)
    p0, p1, p2, p3 = make_pgrids(ph)

    # precompute trace(M) constants for V=S drift: Tr(Hess S) = Tr(M) + 3 lam4 ||A||^2
    # Tr(M) = m2 * n + alpha * Tr(curlcurl) with Tr(curlcurl) = (d-1) * sum_k p2(k)
    with torch.no_grad():
        _, _, p2grid = project_perp_and_curlcurl(A[:1], p0, p1, p2, p3)  # p2grid only
        sum_p2 = float(p2grid.sum().item())
    n_dof = d * (L ** 4)
    trM_const = args.m2 * n_dof + args.alpha * (d - 1) * sum_p2

    # low-k selection (fine)
    with torch.no_grad():
        p2_flat = p2grid.reshape(-1).detach().cpu().numpy()
        mask = p2_flat > 0
        idx_all = np.nonzero(mask)[0]
        idx_sort = idx_all[np.argsort(p2_flat[idx_all])]
        lowk_idx = idx_sort[: int(args.lowk)].astype(np.int64)

    # coarse grids (block2)
    assert L % 2 == 0, "L must be even for 2x blocking"
    Lc = L // 2
    ph_c = make_phat_1d(Lc, device=device, dtype=dtype)
    q0, q1, q2, q3 = make_pgrids(ph_c)

    # precompute coarse trace(M) constants
    with torch.no_grad():
        A0c = block2_avg(A[:1])
        _, _, p2grid_c = project_perp_and_curlcurl(A0c, q0, q1, q2, q3)
        sum_p2_c = float(p2grid_c.sum().item())
    n_dof_c = d * (Lc ** 4)
    trM_const_c = args.m2 * n_dof_c + args.alpha * (d - 1) * sum_p2_c

    # low-k selection (coarse)
    with torch.no_grad():
        p2_flat_c = p2grid_c.reshape(-1).detach().cpu().numpy()
        maskc = p2_flat_c > 0
        idx_all_c = np.nonzero(maskc)[0]
        idx_sort_c = idx_all_c[np.argsort(p2_flat_c[idx_all_c])]
        lowk_idx_c = idx_sort_c[: int(args.lowk)].astype(np.int64)

    # accumulators
    S_list, LV_list, grad2_list, trH_list, norm2_list = [], [], [], [], []
    S_c_list, LV_c_list, grad2_c_list, trH_c_list, norm2_c_list = [], [], [], [], []

    Pk_accum = np.zeros((len(lowk_idx),), dtype=np.float64)
    Pk_accum_c = np.zeros((len(lowk_idx_c),), dtype=np.float64)
    nsamp = 0

    t0 = time.time()

    for t in range(args.steps):
        # gradient step requires curlcurl(A)
        curlcurlA, _, _ = project_perp_and_curlcurl(A, p0, p1, p2, p3)

        gradS = args.m2 * A + args.alpha * curlcurlA + args.lam4 * (A ** 3)

        # Langevin
        A = A - args.dt * gradS + math.sqrt(2.0 * args.dt) * torch.randn_like(A)

        # sample
        if t >= args.burnin and ((t - args.burnin) % args.thin == 0):
            with torch.no_grad():
                # recompute curlcurl + Aperp_hat for stats
                curlcurlA, Aperp_hat, p2grid_now = project_perp_and_curlcurl(A, p0, p1, p2, p3)
                S = energy_S(A, curlcurlA, args.m2, args.alpha, args.lam4)  # [B]
                gradS = args.m2 * A + args.alpha * curlcurlA + args.lam4 * (A ** 3)

                norm2 = (A * A).flatten(1).sum(-1)          # [B]
                grad2 = (gradS * gradS).flatten(1).sum(-1)  # [B]
                trH = trM_const + 3.0 * args.lam4 * norm2   # [B]
                LV = trH - grad2                             # [B]

                # store fine
                S_list.append(S.detach().cpu().numpy())
                LV_list.append(LV.detach().cpu().numpy())
                grad2_list.append(grad2.detach().cpu().numpy())
                trH_list.append(trH.detach().cpu().numpy())
                norm2_list.append(norm2.detach().cpu().numpy())

                # power spectrum on selected low-k (mean over batch,mu)
                # Aperp_hat: [B,d,L^4] complex
                Ap = Aperp_hat.reshape(B, d, -1)
                P = (Ap.real * Ap.real + Ap.imag * Ap.imag).mean(axis=(0, 1))  # [L^4]
                Pk_accum += P[lowk_idx].detach().cpu().numpy()

                # coarse stats via 2x blocking
                Ac = block2_avg(A)
                curlcurlAc, Aperp_hat_c, _ = project_perp_and_curlcurl(Ac, q0, q1, q2, q3)
                Sc = energy_S(Ac, curlcurlAc, args.m2, args.alpha, args.lam4)
                gradSc = args.m2 * Ac + args.alpha * curlcurlAc + args.lam4 * (Ac ** 3)

                norm2c = (Ac * Ac).flatten(1).sum(-1)
                grad2c = (gradSc * gradSc).flatten(1).sum(-1)
                trHc = trM_const_c + 3.0 * args.lam4 * norm2c
                LVc = trHc - grad2c

                S_c_list.append(Sc.detach().cpu().numpy())
                LV_c_list.append(LVc.detach().cpu().numpy())
                grad2_c_list.append(grad2c.detach().cpu().numpy())
                trH_c_list.append(trHc.detach().cpu().numpy())
                norm2_c_list.append(norm2c.detach().cpu().numpy())

                Ap_c = Aperp_hat_c.reshape(B, d, -1)
                Pc = (Ap_c.real * Ap_c.real + Ap_c.imag * Ap_c.imag).mean(axis=(0, 1))  # [Lc^4]
                Pk_accum_c += Pc[lowk_idx_c].detach().cpu().numpy()

                nsamp += 1

        if (t + 1) % max(200, args.thin * 10) == 0:
            dt_wall = time.time() - t0
            print(f"[step {t+1}/{args.steps}] nsamp={nsamp}  wall={dt_wall:.1f}s")

    # concatenate
    S_np = np.concatenate(S_list, axis=0) if S_list else np.empty((0,), dtype=np.float64)
    LV_np = np.concatenate(LV_list, axis=0) if LV_list else np.empty((0,), dtype=np.float64)
    grad2_np = np.concatenate(grad2_list, axis=0) if grad2_list else np.empty((0,), dtype=np.float64)
    trH_np = np.concatenate(trH_list, axis=0) if trH_list else np.empty((0,), dtype=np.float64)
    norm2_np = np.concatenate(norm2_list, axis=0) if norm2_list else np.empty((0,), dtype=np.float64)

    S_c_np = np.concatenate(S_c_list, axis=0) if S_c_list else np.empty((0,), dtype=np.float64)
    LV_c_np = np.concatenate(LV_c_list, axis=0) if LV_c_list else np.empty((0,), dtype=np.float64)
    grad2_c_np = np.concatenate(grad2_c_list, axis=0) if grad2_c_list else np.empty((0,), dtype=np.float64)
    trH_c_np = np.concatenate(trH_c_list, axis=0) if trH_c_list else np.empty((0,), dtype=np.float64)
    norm2_c_np = np.concatenate(norm2_c_list, axis=0) if norm2_c_list else np.empty((0,), dtype=np.float64)

    # average power spectra on low-k sets
    if nsamp > 0:
        Pk_mean = Pk_accum / nsamp
        Pk_mean_c = Pk_accum_c / nsamp
    else:
        Pk_mean = Pk_accum * np.nan
        Pk_mean_c = Pk_accum_c * np.nan

    # drift fits
    fit_f = one_sided_drift_fit(S_np, LV_np, lam_max=args.lam_max, lam_grid=args.lam_grid, q=args.qfit)
    fit_c = one_sided_drift_fit(S_c_np, LV_c_np, lam_max=args.lam_max, lam_grid=args.lam_grid, q=args.qfit)

    # low-k mass fits
    mf_f = lowk_mass_fit(Pk_mean, p2_flat, lowk_idx)
    mf_c = lowk_mass_fit(Pk_mean_c, p2_flat_c, lowk_idx_c)

    # report
    print("\n================= REPORT =================")
    print(f"device={device} dtype={args.dtype}  L={L}  batch={B}  samples={int(S_np.size)}  nsamp_steps={nsamp}")
    if S_np.size > 0:
        print(f"[fine]  S: mean={S_np.mean():.6g}  q50={np.quantile(S_np,0.5):.6g}  q90={np.quantile(S_np,0.9):.6g}")
        print(f"[fine]  LV: mean={LV_np.mean():.6g}  q50={np.quantile(LV_np,0.5):.6g}  q90={np.quantile(LV_np,0.9):.6g}")
    print(f"[fine drift fit] lam*={fit_f['lam_star']:.6g}  b*={fit_f['b_star']:.6g}  score*={fit_f['score_star']:.6g}  viol@*={fit_f['viol_grid'][np.argmax(fit_f['score_grid'])]*100:.3f}%")
    print(f"[fine low-k fit]  m_eff^2={mf_f['m_eff2']:.6g}  Z={mf_f['Z']:.6g}  n={mf_f['n']}")

    if S_c_np.size > 0:
        print(f"[block] S: mean={S_c_np.mean():.6g}  q50={np.quantile(S_c_np,0.5):.6g}  q90={np.quantile(S_c_np,0.9):.6g}")
        print(f"[block] LV: mean={LV_c_np.mean():.6g}  q50={np.quantile(LV_c_np,0.5):.6g}  q90={np.quantile(LV_c_np,0.9):.6g}")
    print(f"[block drift fit] lam*={fit_c['lam_star']:.6g}  b*={fit_c['b_star']:.6g}  score*={fit_c['score_star']:.6g}  viol@*={fit_c['viol_grid'][np.argmax(fit_c['score_grid'])]*100:.3f}%")
    print(f"[block low-k fit]  m_eff^2={mf_c['m_eff2']:.6g}  Z={mf_c['Z']:.6g}  n={mf_c['n']}")
    print("==========================================\n")

    # save
    np.savez(
        args.out,
        # fine
        S=S_np, LV=LV_np, grad2=grad2_np, traceH=trH_np, norm2=norm2_np,
        # coarse
        S_block=S_c_np, LV_block=LV_c_np, grad2_block=grad2_c_np, traceH_block=trH_c_np, norm2_block=norm2_c_np,
        # drift fit grids
        lam_grid=fit_f["lam_grid"], b_grid=fit_f["b_grid"], viol_grid=fit_f["viol_grid"], score_grid=fit_f["score_grid"],
        lam_star=fit_f["lam_star"], b_star=fit_f["b_star"], score_star=fit_f["score_star"],
        lam_grid_block=fit_c["lam_grid"], b_grid_block=fit_c["b_grid"], viol_grid_block=fit_c["viol_grid"], score_grid_block=fit_c["score_grid"],
        lam_star_block=fit_c["lam_star"], b_star_block=fit_c["b_star"], score_star_block=fit_c["score_star"],
        # low-k fits
        lowk_idx=lowk_idx, lowk_idx_block=lowk_idx_c,
        Pk_mean=Pk_mean, Pk_mean_block=Pk_mean_c,
        p2_flat=p2_flat, p2_flat_block=p2_flat_c,
        m_eff2=mf_f["m_eff2"], Z=mf_f["Z"], n_lowk=mf_f["n"],
        m_eff2_block=mf_c["m_eff2"], Z_block=mf_c["Z"], n_lowk_block=mf_c["n"],
        # params
        L=L, L_block=Lc, d=d, batch=B,
        m2=args.m2, alpha=args.alpha, lam4=args.lam4,
        dt=args.dt, steps=args.steps, burnin=args.burnin, thin=args.thin,
        qfit=args.qfit, lam_max=args.lam_max, lam_grid_n=args.lam_grid,
        seed=args.seed,
    )

    print(f"[saved] {args.out}")


if __name__ == "__main__":
    main()


[step 200/6000] nsamp=0  wall=2.3s
[step 400/6000] nsamp=0  wall=4.5s
[step 600/6000] nsamp=0  wall=6.7s
[step 800/6000] nsamp=0  wall=8.9s
[step 1000/6000] nsamp=0  wall=11.1s
[step 1200/6000] nsamp=0  wall=13.3s
[step 1400/6000] nsamp=0  wall=15.5s
[step 1600/6000] nsamp=5  wall=17.8s
[step 1800/6000] nsamp=15  wall=20.1s
[step 2000/6000] nsamp=25  wall=22.5s
[step 2200/6000] nsamp=35  wall=24.9s
[step 2400/6000] nsamp=45  wall=27.2s
[step 2600/6000] nsamp=55  wall=29.6s
[step 2800/6000] nsamp=65  wall=31.9s
[step 3000/6000] nsamp=75  wall=34.3s
[step 3200/6000] nsamp=85  wall=36.7s
[step 3400/6000] nsamp=95  wall=39.0s
[step 3600/6000] nsamp=105  wall=41.4s
[step 3800/6000] nsamp=115  wall=43.7s
[step 4000/6000] nsamp=125  wall=46.1s
[step 4200/6000] nsamp=135  wall=48.5s
[step 4400/6000] nsamp=145  wall=50.8s
[step 4600/6000] nsamp=155  wall=53.2s
[step 4800/6000] nsamp=165  wall=55.6s
[step 5000/6000] nsamp=175  wall=57.9s
[step 5200/6000] nsamp=185  wall=60.3s
[step 5400/6000] ns

In [5]:
# ============================================================
# UNIFIED GPU RUN v2 (A100-ready): 4D 1-form "Maxwell + quartic" Langevin
#
# Fixes from v1:
#  (A) Correct coarse momentum for 2x blocking: a' = 2 => p̂_coarse = (1/2) * p̂(L/2)
#  (B) Add RG-motivated field rescaling for 1-forms in 4D: A' ≈ 2 * block_avg(A)
#  (C) Drift uses intensive Lyapunov V2 = ||A||^2 / n_dof
#        L V2 = 2 - (2/n) <∇S, A>
#      which is scale-stable and not dominated by Tr(M).
#
# Outputs:
#   - NPZ with fine + (naive block, RG block) stats
#   - one-sided drift fits for V2
#   - low-k inverse propagator fits: 1/G(k) ~ m_eff^2 + Z p̂^2
#
# This is NOT Yang–Mills. It is a gauge-adjacent interacting 1-form sandbox.
# ============================================================

import math
import time
import argparse
import numpy as np
import torch
import torch.fft as fft


def set_seed(seed: int):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--L", type=int, default=32)
    p.add_argument("--d", type=int, default=4)
    p.add_argument("--batch", type=int, default=8)

    p.add_argument("--m2", type=float, default=0.30)
    p.add_argument("--alpha", type=float, default=1.0)
    p.add_argument("--lam4", type=float, default=0.50)

    p.add_argument("--dt", type=float, default=5e-4)
    p.add_argument("--steps", type=int, default=6000)
    p.add_argument("--burnin", type=int, default=1500)
    p.add_argument("--thin", type=int, default=20)

    p.add_argument("--init_sigma", type=float, default=0.20)
    p.add_argument("--dtype", type=str, default="float32", choices=["float32", "float64"])
    p.add_argument("--device", type=str, default="cuda")

    p.add_argument("--lowk", type=int, default=256, help="number of smallest nonzero p^2 modes for mass fit")
    p.add_argument("--qfit", type=float, default=0.995, help="quantile for one-sided drift fit")
    p.add_argument("--lam_max", type=float, default=2.0, help="max lambda grid for drift fit (intensive V2)")
    p.add_argument("--lam_grid", type=int, default=401, help="lambda grid points for drift fit")

    # RG toggles for blocked field
    p.add_argument("--rg_field_scale", type=float, default=2.0, help="field rescale after 2x block (4D 1-form ~2)")
    p.add_argument("--out", type=str, default="unified_maxwell_phi4_drift_block_v2.npz")
    p.add_argument("--seed", type=int, default=0)

    args, _ = p.parse_known_args()
    return args


# ---------- FFT momentum (with explicit lattice spacing a) ----------
def make_phat_1d(L: int, a: float, device: str, dtype: torch.dtype):
    # p̂(n) = (2/a) sin(pi n / L)  (since p = 2π n/L, sin(p a/2)=sin(pi n a / L) and here a=1 or 2)
    # For coarse after 2x blocking: a' = 2 => p̂ is half the a=1 convention.
    f = torch.fft.fftfreq(L, d=1.0, device=device, dtype=dtype)  # n/L with wrap
    return (2.0 / a) * torch.sin(math.pi * f)


def make_pgrids(ph: torch.Tensor):
    L = ph.numel()
    p0 = ph.view(L, 1, 1, 1)
    p1 = ph.view(1, L, 1, 1)
    p2 = ph.view(1, 1, L, 1)
    p3 = ph.view(1, 1, 1, L)
    return p0, p1, p2, p3


def project_perp_and_curlcurl(A: torch.Tensor, p0, p1, p2, p3, eps=1e-12):
    """
    A: [B,d,L,L,L,L] real
    returns:
      curlcurlA: [B,d,L,L,L,L] real
      Aperp_hat: [B,d,L,L,L,L] complex
      p2grid: [L,L,L,L] real
    """
    spatial = (-4, -3, -2, -1)
    Ahat = fft.fftn(A, dim=spatial, norm="ortho")  # complex

    p2grid = (p0 * p0 + p1 * p1 + p2 * p2 + p3 * p3)  # [L,L,L,L]

    dot = (p0 * Ahat[:, 0] + p1 * Ahat[:, 1] + p2 * Ahat[:, 2] + p3 * Ahat[:, 3])  # [B,L^4] complex

    inv = torch.zeros_like(p2grid)
    mask = p2grid > 0
    inv[mask] = 1.0 / (p2grid[mask] + eps)

    factor = dot * inv
    Aperp_hat = torch.stack(
        [
            Ahat[:, 0] - p0 * factor,
            Ahat[:, 1] - p1 * factor,
            Ahat[:, 2] - p2 * factor,
            Ahat[:, 3] - p3 * factor,
        ],
        dim=1,
    )

    curlcurl_hat = Aperp_hat * p2grid
    curlcurlA = fft.ifftn(curlcurl_hat, dim=spatial, norm="ortho").real
    return curlcurlA, Aperp_hat, p2grid


def energy_S(A, curlcurlA, m2, alpha, lam4):
    mass = 0.5 * m2 * (A * A).flatten(1).sum(-1)
    curl = 0.5 * alpha * (A * curlcurlA).flatten(1).sum(-1)
    quart = 0.25 * lam4 * (A ** 4).flatten(1).sum(-1)
    return mass + curl + quart


def block2_avg(A):
    B, d, L, _, _, _ = A.shape
    assert L % 2 == 0
    L2 = L // 2
    return A.reshape(B, d, L2, 2, L2, 2, L2, 2, L2, 2).mean(dim=(3, 5, 7, 9))


def one_sided_drift_fit(V_np, LV_np, lam_max=2.0, lam_grid=401, q=0.995):
    V = np.asarray(V_np, dtype=np.float64)
    LV = np.asarray(LV_np, dtype=np.float64)
    lam_vals = np.linspace(0.0, float(lam_max), int(lam_grid))
    b_vals = np.empty_like(lam_vals)
    viol_vals = np.empty_like(lam_vals)
    score_vals = np.empty_like(lam_vals)

    eps = 1e-12
    for i, lam in enumerate(lam_vals):
        z = LV + lam * V
        b = np.quantile(z, q)
        b_vals[i] = b
        viol_vals[i] = np.mean(LV > (-lam * V + b))
        # intensive V => b should be O(1), score meaningful
        score_vals[i] = lam / (b + eps)

    i_star = int(np.argmax(score_vals))
    return dict(
        lam_star=float(lam_vals[i_star]),
        b_star=float(b_vals[i_star]),
        score_star=float(score_vals[i_star]),
        lam_grid=lam_vals,
        b_grid=b_vals,
        viol_grid=viol_vals,
        score_grid=score_vals,
    )


def lowk_mass_fit(power_k, p2_flat, lowk_idx):
    G = np.asarray(power_k, dtype=np.float64)
    p2 = np.asarray(p2_flat[lowk_idx], dtype=np.float64)

    good = (G > 0) & np.isfinite(G) & np.isfinite(p2)
    G = G[good]
    p2 = p2[good]
    if G.size < 8:
        return dict(m_eff2=np.nan, Z=np.nan, n=int(G.size))

    y = 1.0 / G
    x = p2
    A = np.stack([np.ones_like(x), x], axis=1)
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    return dict(m_eff2=float(coef[0]), Z=float(coef[1]), n=int(G.size))


def pick_lowk(p2grid_torch, K):
    p2_flat = p2grid_torch.reshape(-1).detach().cpu().numpy()
    mask = p2_flat > 0
    idx_all = np.nonzero(mask)[0]
    idx_sort = idx_all[np.argsort(p2_flat[idx_all])]
    return p2_flat, idx_sort[: int(K)].astype(np.int64)


def main():
    args = parse_args()

    if args.device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA requested but not available")

    device = args.device
    dtype = torch.float32 if args.dtype == "float32" else torch.float64
    set_seed(args.seed)

    L, d, B = args.L, args.d, args.batch
    assert d == 4, "this script assumes d=4"
    assert L % 2 == 0, "L must be even for 2x blocking"

    # init field
    A = (args.init_sigma * torch.randn((B, d, L, L, L, L), device=device, dtype=dtype))

    # fine momentum grids (a=1)
    ph = make_phat_1d(L, a=1.0, device=device, dtype=dtype)
    p0, p1, p2, p3 = make_pgrids(ph)

    # coarse momentum grids (a'=2)
    Lc = L // 2
    ph_c = make_phat_1d(Lc, a=2.0, device=device, dtype=dtype)  # <-- KEY FIX: a'=2
    q0, q1, q2, q3 = make_pgrids(ph_c)

    # low-k indices fine/coarse
    with torch.no_grad():
        curl0, Ahat0, p2grid = project_perp_and_curlcurl(A[:1], p0, p1, p2, p3)
        p2_flat, lowk_idx = pick_lowk(p2grid, args.lowk)

        A0c_naive = block2_avg(A[:1])
        curlc0, Ahatc0, p2grid_c = project_perp_and_curlcurl(A0c_naive, q0, q1, q2, q3)
        p2_flat_c, lowk_idx_c = pick_lowk(p2grid_c, args.lowk)

    # accumulators
    # fine
    S_list, V2_list, LV2_list = [], [], []
    Pk_accum = np.zeros((len(lowk_idx),), dtype=np.float64)

    # coarse naive
    Sbn_list, V2bn_list, LV2bn_list = [], [], []
    Pk_bn_accum = np.zeros((len(lowk_idx_c),), dtype=np.float64)

    # coarse RG-rescaled field
    Sbr_list, V2br_list, LV2br_list = [], [], []
    Pk_br_accum = np.zeros((len(lowk_idx_c),), dtype=np.float64)

    nsamp = 0
    t0 = time.time()

    n_f = d * (L ** 4)
    n_c = d * (Lc ** 4)

    for t in range(args.steps):
        # fine dynamics
        curlcurlA, _, _ = project_perp_and_curlcurl(A, p0, p1, p2, p3)
        gradS = args.m2 * A + args.alpha * curlcurlA + args.lam4 * (A ** 3)
        A = A - args.dt * gradS + math.sqrt(2.0 * args.dt) * torch.randn_like(A)

        # sample
        if t >= args.burnin and ((t - args.burnin) % args.thin == 0):
            with torch.no_grad():
                # -------- fine stats --------
                curlcurlA, Aperp_hat, _ = project_perp_and_curlcurl(A, p0, p1, p2, p3)
                S = energy_S(A, curlcurlA, args.m2, args.alpha, args.lam4)
                gradS = args.m2 * A + args.alpha * curlcurlA + args.lam4 * (A ** 3)

                V2 = (A * A).flatten(1).sum(-1) / float(n_f)
                inner = (gradS * A).flatten(1).sum(-1)
                LV2 = 2.0 - (2.0 / float(n_f)) * inner

                S_list.append(S.detach().cpu().numpy())
                V2_list.append(V2.detach().cpu().numpy())
                LV2_list.append(LV2.detach().cpu().numpy())

                Ap = Aperp_hat.reshape(B, d, -1)
                P = (Ap.real * Ap.real + Ap.imag * Ap.imag).mean(dim=(0, 1))
                Pk_accum += P[lowk_idx].detach().cpu().numpy()

                # -------- coarse naive block (avg) --------
                Abn = block2_avg(A)
                curlbn, Aperp_bn_hat, _ = project_perp_and_curlcurl(Abn, q0, q1, q2, q3)
                Sbn = energy_S(Abn, curlbn, args.m2, args.alpha, args.lam4)
                gradSbn = args.m2 * Abn + args.alpha * curlbn + args.lam4 * (Abn ** 3)

                V2bn = (Abn * Abn).flatten(1).sum(-1) / float(n_c)
                innerbn = (gradSbn * Abn).flatten(1).sum(-1)
                LV2bn = 2.0 - (2.0 / float(n_c)) * innerbn

                Sbn_list.append(Sbn.detach().cpu().numpy())
                V2bn_list.append(V2bn.detach().cpu().numpy())
                LV2bn_list.append(LV2bn.detach().cpu().numpy())

                Apbn = Aperp_bn_hat.reshape(B, d, -1)
                Pbn = (Apbn.real * Apbn.real + Apbn.imag * Apbn.imag).mean(dim=(0, 1))
                Pk_bn_accum += Pbn[lowk_idx_c].detach().cpu().numpy()

                # -------- coarse RG block (field rescale) --------
                Abr = args.rg_field_scale * Abn
                curlbr, Aperp_br_hat, _ = project_perp_and_curlcurl(Abr, q0, q1, q2, q3)
                Sbr = energy_S(Abr, curlbr, args.m2, args.alpha, args.lam4)
                gradSbr = args.m2 * Abr + args.alpha * curlbr + args.lam4 * (Abr ** 3)

                V2br = (Abr * Abr).flatten(1).sum(-1) / float(n_c)
                innerbr = (gradSbr * Abr).flatten(1).sum(-1)
                LV2br = 2.0 - (2.0 / float(n_c)) * innerbr

                Sbr_list.append(Sbr.detach().cpu().numpy())
                V2br_list.append(V2br.detach().cpu().numpy())
                LV2br_list.append(LV2br.detach().cpu().numpy())

                Apbr = Aperp_br_hat.reshape(B, d, -1)
                Pbr = (Apbr.real * Apbr.real + Apbr.imag * Apbr.imag).mean(dim=(0, 1))
                Pk_br_accum += Pbr[lowk_idx_c].detach().cpu().numpy()

                nsamp += 1

        if (t + 1) % max(200, args.thin * 10) == 0:
            print(f"[step {t+1}/{args.steps}] nsamp={nsamp}  wall={time.time()-t0:.1f}s")

    # concat
    def cat(xs):
        return np.concatenate(xs, axis=0) if xs else np.empty((0,), dtype=np.float64)

    S_f = cat(S_list)
    V2_f = cat(V2_list)
    LV2_f = cat(LV2_list)

    S_bn = cat(Sbn_list)
    V2_bn = cat(V2bn_list)
    LV2_bn = cat(LV2bn_list)

    S_br = cat(Sbr_list)
    V2_br = cat(V2br_list)
    LV2_br = cat(LV2br_list)

    # mean spectra
    if nsamp > 0:
        Pk_f = Pk_accum / nsamp
        Pk_bn = Pk_bn_accum / nsamp
        Pk_br = Pk_br_accum / nsamp
    else:
        Pk_f = Pk_accum * np.nan
        Pk_bn = Pk_bn_accum * np.nan
        Pk_br = Pk_br_accum * np.nan

    # fits
    fit_f = one_sided_drift_fit(V2_f, LV2_f, lam_max=args.lam_max, lam_grid=args.lam_grid, q=args.qfit)
    fit_bn = one_sided_drift_fit(V2_bn, LV2_bn, lam_max=args.lam_max, lam_grid=args.lam_grid, q=args.qfit)
    fit_br = one_sided_drift_fit(V2_br, LV2_br, lam_max=args.lam_max, lam_grid=args.lam_grid, q=args.qfit)

    mf_f = lowk_mass_fit(Pk_f, p2_flat, lowk_idx)
    mf_bn = lowk_mass_fit(Pk_bn, p2_flat_c, lowk_idx_c)
    mf_br = lowk_mass_fit(Pk_br, p2_flat_c, lowk_idx_c)

    # report
    print("\n================= REPORT v2 ================")
    print(f"device={device} dtype={args.dtype}  L={L} -> Lc={Lc}  batch={B}  nsamp={nsamp}  samples={int(S_f.size)}")
    print(f"params: m2={args.m2} alpha={args.alpha} lam4={args.lam4} dt={args.dt} burnin={args.burnin} thin={args.thin}")
    print(f"RG: field_scale={args.rg_field_scale}   (coarse p̂ uses a'=2 by construction)")

    if S_f.size > 0:
        print(f"[fine]   V2 mean={V2_f.mean():.6g}  LV2 mean={LV2_f.mean():.6g}  q90(LV2)={np.quantile(LV2_f,0.9):.6g}")
    print(f"[fine drift]   lam*={fit_f['lam_star']:.6g}  b*={fit_f['b_star']:.6g}  viol@*={fit_f['viol_grid'][np.argmax(fit_f['score_grid'])]*100:.3f}%")
    print(f"[fine low-k]   m_eff^2={mf_f['m_eff2']:.6g}  Z={mf_f['Z']:.6g}  n={mf_f['n']}")

    if S_bn.size > 0:
        print(f"[block naive] V2 mean={V2_bn.mean():.6g}  LV2 mean={LV2_bn.mean():.6g}  q90(LV2)={np.quantile(LV2_bn,0.9):.6g}")
    print(f"[bn drift]     lam*={fit_bn['lam_star']:.6g}  b*={fit_bn['b_star']:.6g}  viol@*={fit_bn['viol_grid'][np.argmax(fit_bn['score_grid'])]*100:.3f}%")
    print(f"[bn low-k]     m_eff^2={mf_bn['m_eff2']:.6g}  Z={mf_bn['Z']:.6g}  n={mf_bn['n']}")

    if S_br.size > 0:
        print(f"[block RG]    V2 mean={V2_br.mean():.6g}  LV2 mean={LV2_br.mean():.6g}  q90(LV2)={np.quantile(LV2_br,0.9):.6g}")
    print(f"[br drift]     lam*={fit_br['lam_star']:.6g}  b*={fit_br['b_star']:.6g}  viol@*={fit_br['viol_grid'][np.argmax(fit_br['score_grid'])]*100:.3f}%")
    print(f"[br low-k]     m_eff^2={mf_br['m_eff2']:.6g}  Z={mf_br['Z']:.6g}  n={mf_br['n']}")
    print("============================================\n")

    np.savez(
        args.out,
        # fine
        S=S_f, V2=V2_f, LV2=LV2_f,
        Pk_mean=Pk_f, lowk_idx=lowk_idx, p2_flat=p2_flat,
        # coarse naive
        S_block_naive=S_bn, V2_block_naive=V2_bn, LV2_block_naive=LV2_bn,
        Pk_mean_block_naive=Pk_bn, lowk_idx_block=lowk_idx_c, p2_flat_block=p2_flat_c,
        # coarse RG
        S_block_rg=S_br, V2_block_rg=V2_br, LV2_block_rg=LV2_br,
        Pk_mean_block_rg=Pk_br,
        # drift fits
        lam_grid=fit_f["lam_grid"], b_grid=fit_f["b_grid"], viol_grid=fit_f["viol_grid"], score_grid=fit_f["score_grid"],
        lam_star=fit_f["lam_star"], b_star=fit_f["b_star"], score_star=fit_f["score_star"],
        lam_grid_bn=fit_bn["lam_grid"], b_grid_bn=fit_bn["b_grid"], viol_grid_bn=fit_bn["viol_grid"], score_grid_bn=fit_bn["score_grid"],
        lam_star_bn=fit_bn["lam_star"], b_star_bn=fit_bn["b_star"], score_star_bn=fit_bn["score_star"],
        lam_grid_br=fit_br["lam_grid"], b_grid_br=fit_br["b_grid"], viol_grid_br=fit_br["viol_grid"], score_grid_br=fit_br["score_grid"],
        lam_star_br=fit_br["lam_star"], b_star_br=fit_br["b_star"], score_star_br=fit_br["score_star"],
        # low-k fits
        m_eff2=mf_f["m_eff2"], Z=mf_f["Z"], n_lowk=mf_f["n"],
        m_eff2_bn=mf_bn["m_eff2"], Z_bn=mf_bn["Z"], n_lowk_bn=mf_bn["n"],
        m_eff2_br=mf_br["m_eff2"], Z_br=mf_br["Z"], n_lowk_br=mf_br["n"],
        # params
        L=L, L_block=Lc, d=d, batch=B,
        m2=args.m2, alpha=args.alpha, lam4=args.lam4,
        dt=args.dt, steps=args.steps, burnin=args.burnin, thin=args.thin,
        qfit=args.qfit, lam_max=args.lam_max, lam_grid_n=args.lam_grid,
        seed=args.seed,
        rg_field_scale=args.rg_field_scale,
    )

    print(f"[saved] {args.out}")


if __name__ == "__main__":
    main()


[step 200/6000] nsamp=0  wall=2.2s
[step 400/6000] nsamp=0  wall=4.4s
[step 600/6000] nsamp=0  wall=6.6s
[step 800/6000] nsamp=0  wall=8.8s
[step 1000/6000] nsamp=0  wall=11.0s
[step 1200/6000] nsamp=0  wall=13.2s
[step 1400/6000] nsamp=0  wall=15.4s
[step 1600/6000] nsamp=5  wall=17.7s
[step 1800/6000] nsamp=15  wall=20.1s
[step 2000/6000] nsamp=25  wall=22.5s
[step 2200/6000] nsamp=35  wall=24.8s
[step 2400/6000] nsamp=45  wall=27.2s
[step 2600/6000] nsamp=55  wall=29.6s
[step 2800/6000] nsamp=65  wall=32.0s
[step 3000/6000] nsamp=75  wall=34.4s
[step 3200/6000] nsamp=85  wall=36.7s
[step 3400/6000] nsamp=95  wall=39.1s
[step 3600/6000] nsamp=105  wall=41.5s
[step 3800/6000] nsamp=115  wall=43.9s
[step 4000/6000] nsamp=125  wall=46.3s
[step 4200/6000] nsamp=135  wall=48.6s
[step 4400/6000] nsamp=145  wall=51.0s
[step 4600/6000] nsamp=155  wall=53.4s
[step 4800/6000] nsamp=165  wall=55.8s
[step 5000/6000] nsamp=175  wall=58.1s
[step 5200/6000] nsamp=185  wall=60.5s
[step 5400/6000] ns

In [6]:
# ============================================================
# RG CLOSURE TEST (GPU): does block-induced distribution close within same ansatz?
#
# Input: unified_maxwell_phi4_drift_block_v2.npz (from your v2 run)
# Reads:
#   - blocked effective (m_eff2_br, Z_br)
#   - blocked RG field statistics (V2_block_rg, LV2_block_rg, etc.)
#
# Then runs an independent coarse simulation at Lc=L/2 with:
#   m2 = m_eff2_br
#   alpha = Z_br
#   lam4 = (default) same as fine, unless overridden
#
# Compares:
#   - low-k (m_eff^2, Z) coarse sim vs blocked RG
#   - V2 quantiles coarse sim vs blocked RG
#   - drift fit (LV2 <= -lam* V2 + b*) coarse sim vs blocked RG
#
# Saves: rg_closure_report.npz (targets + coarse sim outputs)
# ============================================================

import math
import time
import argparse
import numpy as np
import torch
import torch.fft as fft


def set_seed(seed: int):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--inp", type=str, default="unified_maxwell_phi4_drift_block_v2.npz")
    p.add_argument("--device", type=str, default="cuda")
    p.add_argument("--dtype", type=str, default="float32", choices=["float32", "float64"])

    # coarse sim controls
    p.add_argument("--steps", type=int, default=8000)
    p.add_argument("--burnin", type=int, default=2000)
    p.add_argument("--thin", type=int, default=20)
    p.add_argument("--dt", type=float, default=5e-4)
    p.add_argument("--batch", type=int, default=16)
    p.add_argument("--init_sigma", type=float, default=0.20)

    # fitting controls
    p.add_argument("--lowk", type=int, default=256)
    p.add_argument("--qfit", type=float, default=0.995)
    p.add_argument("--lam_max", type=float, default=5.0)
    p.add_argument("--lam_grid", type=int, default=501)

    # optional override
    p.add_argument("--override_m2", type=float, default=np.nan)
    p.add_argument("--override_alpha", type=float, default=np.nan)
    p.add_argument("--override_lam4", type=float, default=np.nan)

    p.add_argument("--out", type=str, default="rg_closure_report.npz")
    p.add_argument("--seed", type=int, default=0)

    args, _ = p.parse_known_args()
    return args


def make_phat_1d(L: int, a: float, device: str, dtype: torch.dtype):
    f = torch.fft.fftfreq(L, d=1.0, device=device, dtype=dtype)
    return (2.0 / a) * torch.sin(math.pi * f)


def make_pgrids(ph: torch.Tensor):
    L = ph.numel()
    p0 = ph.view(L, 1, 1, 1)
    p1 = ph.view(1, L, 1, 1)
    p2 = ph.view(1, 1, L, 1)
    p3 = ph.view(1, 1, 1, L)
    return p0, p1, p2, p3


def project_perp_and_curlcurl(A: torch.Tensor, p0, p1, p2, p3, eps=1e-12):
    spatial = (-4, -3, -2, -1)
    Ahat = fft.fftn(A, dim=spatial, norm="ortho")  # complex
    p2grid = (p0 * p0 + p1 * p1 + p2 * p2 + p3 * p3)

    dot = (p0 * Ahat[:, 0] + p1 * Ahat[:, 1] + p2 * Ahat[:, 2] + p3 * Ahat[:, 3])

    inv = torch.zeros_like(p2grid)
    mask = p2grid > 0
    inv[mask] = 1.0 / (p2grid[mask] + eps)

    factor = dot * inv
    Aperp_hat = torch.stack(
        [
            Ahat[:, 0] - p0 * factor,
            Ahat[:, 1] - p1 * factor,
            Ahat[:, 2] - p2 * factor,
            Ahat[:, 3] - p3 * factor,
        ],
        dim=1,
    )

    curlcurl_hat = Aperp_hat * p2grid
    curlcurlA = fft.ifftn(curlcurl_hat, dim=spatial, norm="ortho").real
    return curlcurlA, Aperp_hat, p2grid


def energy_S(A, curlcurlA, m2, alpha, lam4):
    mass = 0.5 * m2 * (A * A).flatten(1).sum(-1)
    curl = 0.5 * alpha * (A * curlcurlA).flatten(1).sum(-1)
    quart = 0.25 * lam4 * (A ** 4).flatten(1).sum(-1)
    return mass + curl + quart


def pick_lowk(p2grid_torch, K):
    p2_flat = p2grid_torch.reshape(-1).detach().cpu().numpy()
    mask = p2_flat > 0
    idx_all = np.nonzero(mask)[0]
    idx_sort = idx_all[np.argsort(p2_flat[idx_all])]
    return p2_flat, idx_sort[: int(K)].astype(np.int64)


def one_sided_drift_fit(V_np, LV_np, lam_max=5.0, lam_grid=501, q=0.995):
    V = np.asarray(V_np, dtype=np.float64)
    LV = np.asarray(LV_np, dtype=np.float64)

    lam_vals = np.linspace(0.0, float(lam_max), int(lam_grid))
    b_vals = np.empty_like(lam_vals)
    viol_vals = np.empty_like(lam_vals)
    score_vals = np.empty_like(lam_vals)

    eps = 1e-12
    for i, lam in enumerate(lam_vals):
        z = LV + lam * V
        b = np.quantile(z, q)
        b_vals[i] = b
        viol_vals[i] = np.mean(LV > (-lam * V + b))
        score_vals[i] = lam / (b + eps)

    i_star = int(np.argmax(score_vals))
    return dict(
        lam_star=float(lam_vals[i_star]),
        b_star=float(b_vals[i_star]),
        score_star=float(score_vals[i_star]),
        lam_grid=lam_vals,
        b_grid=b_vals,
        viol_grid=viol_vals,
        score_grid=score_vals,
    )


def lowk_mass_fit(power_k, p2_flat, lowk_idx):
    G = np.asarray(power_k, dtype=np.float64)
    p2 = np.asarray(p2_flat[lowk_idx], dtype=np.float64)

    good = (G > 0) & np.isfinite(G) & np.isfinite(p2)
    G = G[good]
    p2 = p2[good]
    if G.size < 8:
        return dict(m_eff2=np.nan, Z=np.nan, n=int(G.size))

    y = 1.0 / G
    x = p2
    A = np.stack([np.ones_like(x), x], axis=1)
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    return dict(m_eff2=float(coef[0]), Z=float(coef[1]), n=int(G.size))


def run_coarse_sim(Lc, batch, m2, alpha, lam4, dt, steps, burnin, thin, lowkK, qfit, lam_max, lam_grid, device, dtype):
    d = 4
    a = 2.0  # coarse spacing corresponding to your construction

    ph = make_phat_1d(Lc, a=a, device=device, dtype=dtype)
    p0, p1, p2, p3 = make_pgrids(ph)

    Afield = 0.20 * torch.randn((batch, d, Lc, Lc, Lc, Lc), device=device, dtype=dtype)

    with torch.no_grad():
        _, _, p2grid = project_perp_and_curlcurl(Afield[:1], p0, p1, p2, p3)
        p2_flat, lowk_idx = pick_lowk(p2grid, lowkK)

    n = float(d * (Lc ** 4))

    V2_list, LV2_list = [], []
    Pk_accum = np.zeros((len(lowk_idx),), dtype=np.float64)
    nsamp = 0
    t0 = time.time()

    for t in range(steps):
        curlcurlA, _, _ = project_perp_and_curlcurl(Afield, p0, p1, p2, p3)
        gradS = m2 * Afield + alpha * curlcurlA + lam4 * (Afield ** 3)

        Afield = Afield - dt * gradS + math.sqrt(2.0 * dt) * torch.randn_like(Afield)

        if t >= burnin and ((t - burnin) % thin == 0):
            with torch.no_grad():
                curlcurlA, Aperp_hat, _ = project_perp_and_curlcurl(Afield, p0, p1, p2, p3)
                gradS = m2 * Afield + alpha * curlcurlA + lam4 * (Afield ** 3)

                V2 = (Afield * Afield).flatten(1).sum(-1) / n
                inner = (gradS * Afield).flatten(1).sum(-1)
                LV2 = 2.0 - (2.0 / n) * inner

                V2_list.append(V2.detach().cpu().numpy())
                LV2_list.append(LV2.detach().cpu().numpy())

                Ap = Aperp_hat.reshape(batch, d, -1)
                P = (Ap.real * Ap.real + Ap.imag * Ap.imag).mean(dim=(0, 1))
                Pk_accum += P[lowk_idx].detach().cpu().numpy()

                nsamp += 1

        if (t + 1) % max(400, thin * 20) == 0:
            print(f"[coarse step {t+1}/{steps}] nsamp={nsamp} wall={time.time()-t0:.1f}s")

    V2_np = np.concatenate(V2_list, axis=0) if V2_list else np.empty((0,), dtype=np.float64)
    LV2_np = np.concatenate(LV2_list, axis=0) if LV2_list else np.empty((0,), dtype=np.float64)
    Pk_mean = (Pk_accum / nsamp) if nsamp > 0 else Pk_accum * np.nan

    mf = lowk_mass_fit(Pk_mean, p2_flat, lowk_idx)
    fit = one_sided_drift_fit(V2_np, LV2_np, lam_max=lam_max, lam_grid=lam_grid, q=qfit)

    return dict(
        V2=V2_np, LV2=LV2_np,
        Pk_mean=Pk_mean, p2_flat=p2_flat, lowk_idx=lowk_idx,
        m_eff2=mf["m_eff2"], Z=mf["Z"], n_lowk=mf["n"],
        lam_star=fit["lam_star"], b_star=fit["b_star"], score_star=fit["score_star"],
        lam_grid=fit["lam_grid"], b_grid=fit["b_grid"], viol_grid=fit["viol_grid"], score_grid=fit["score_grid"],
        nsamp=nsamp,
    )


def qstats(x):
    x = np.asarray(x, dtype=np.float64)
    if x.size == 0:
        return dict(mean=np.nan, q50=np.nan, q90=np.nan, q99=np.nan)
    return dict(
        mean=float(x.mean()),
        q50=float(np.quantile(x, 0.5)),
        q90=float(np.quantile(x, 0.9)),
        q99=float(np.quantile(x, 0.99)),
    )


def main():
    args = parse_args()

    if args.device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA requested but not available")

    device = args.device
    dtype = torch.float32 if args.dtype == "float32" else torch.float64
    set_seed(args.seed)

    data = np.load(args.inp, allow_pickle=False)

    L = int(data["L"])
    Lc = int(data["L_block"])
    lam4_f = float(data["lam4"]) if "lam4" in data else 0.5

    # target from blocked RG field
    V2_blk = data["V2_block_rg"]
    LV2_blk = data["LV2_block_rg"]
    targ_V2 = qstats(V2_blk)
    targ_LV2 = qstats(LV2_blk)

    targ_m_eff2 = float(data["m_eff2_br"])
    targ_Z = float(data["Z_br"])

    # choose coarse sim params
    m2 = targ_m_eff2 if np.isnan(args.override_m2) else float(args.override_m2)
    alpha = targ_Z if np.isnan(args.override_alpha) else float(args.override_alpha)
    lam4 = lam4_f if np.isnan(args.override_lam4) else float(args.override_lam4)

    print("\n================= RG CLOSURE TARGETS =================")
    print(f"from file: L={L} -> Lc={Lc}")
    print(f"[blocked RG low-k] m_eff^2={targ_m_eff2:.6g}  Z={targ_Z:.6g}")
    print(f"[blocked RG V2]   mean={targ_V2['mean']:.6g} q50={targ_V2['q50']:.6g} q90={targ_V2['q90']:.6g} q99={targ_V2['q99']:.6g}")
    print(f"[blocked RG LV2]  mean={targ_LV2['mean']:.6g} q50={targ_LV2['q50']:.6g} q90={targ_LV2['q90']:.6g} q99={targ_LV2['q99']:.6g}")
    print("======================================================\n")

    print("================= COARSE SIM SETUP =================")
    print(f"device={device} dtype={args.dtype} batch={args.batch}")
    print(f"m2={m2:.6g} alpha={alpha:.6g} lam4={lam4:.6g}  dt={args.dt} steps={args.steps} burnin={args.burnin} thin={args.thin}")
    print("====================================================\n")

    out = run_coarse_sim(
        Lc=Lc, batch=args.batch,
        m2=m2, alpha=alpha, lam4=lam4,
        dt=args.dt, steps=args.steps, burnin=args.burnin, thin=args.thin,
        lowkK=args.lowk, qfit=args.qfit, lam_max=args.lam_max, lam_grid=args.lam_grid,
        device=device, dtype=dtype
    )

    sim_V2 = qstats(out["V2"])
    sim_LV2 = qstats(out["LV2"])

    print("\n================= COARSE SIM REPORT =================")
    print(f"[coarse sim low-k] m_eff^2={out['m_eff2']:.6g}  Z={out['Z']:.6g}  n={out['n_lowk']}")
    print(f"[coarse sim V2]    mean={sim_V2['mean']:.6g} q50={sim_V2['q50']:.6g} q90={sim_V2['q90']:.6g} q99={sim_V2['q99']:.6g}")
    print(f"[coarse sim LV2]   mean={sim_LV2['mean']:.6g} q50={sim_LV2['q50']:.6g} q90={sim_LV2['q90']:.6g} q99={sim_LV2['q99']:.6g}")
    print(f"[coarse drift fit] lam*={out['lam_star']:.6g}  b*={out['b_star']:.6g}  viol@*={out['viol_grid'][np.argmax(out['score_grid'])]*100:.3f}%")
    print("=====================================================\n")

    # closure errors
    def relerr(a, b):
        return float(abs(a - b) / (abs(b) + 1e-12))

    e_m = relerr(out["m_eff2"], targ_m_eff2)
    e_Z = relerr(out["Z"], targ_Z)
    e_V2 = relerr(sim_V2["mean"], targ_V2["mean"])

    print("================= CLOSURE ERRORS =================")
    print(f"relerr m_eff^2 : {e_m*100:.3f}%")
    print(f"relerr Z       : {e_Z*100:.3f}%")
    print(f"relerr mean(V2): {e_V2*100:.3f}%")
    print("===============================================\n")

    np.savez(
        args.out,
        # targets
        L=L, Lc=Lc,
        target_m_eff2=targ_m_eff2, target_Z=targ_Z,
        target_V2=V2_blk, target_LV2=LV2_blk,
        # sim params
        sim_m2=m2, sim_alpha=alpha, sim_lam4=lam4,
        dt=args.dt, steps=args.steps, burnin=args.burnin, thin=args.thin,
        # sim outputs
        sim_V2=out["V2"], sim_LV2=out["LV2"],
        sim_Pk_mean=out["Pk_mean"], sim_p2_flat=out["p2_flat"], sim_lowk_idx=out["lowk_idx"],
        sim_m_eff2=out["m_eff2"], sim_Z=out["Z"],
        sim_lam_star=out["lam_star"], sim_b_star=out["b_star"], sim_score_star=out["score_star"],
        sim_lam_grid=out["lam_grid"], sim_b_grid=out["b_grid"], sim_viol_grid=out["viol_grid"], sim_score_grid=out["score_grid"],
        # errors
        relerr_m_eff2=e_m, relerr_Z=e_Z, relerr_V2_mean=e_V2,
        seed=args.seed,
    )
    print(f"[saved] {args.out}")


if __name__ == "__main__":
    main()



================= RG CLOSURE TARGETS =================
from file: L=32 -> Lc=16
[blocked RG low-k] m_eff^2=4.57387  Z=6.19872
[blocked RG V2]   mean=0.121648 q50=0.123716 q90=0.124965 q99=0.125402
[blocked RG LV2]  mean=1.58524 q50=1.5788 q90=1.6087 q99=1.62836

================= COARSE SIM SETUP =================
device=cuda dtype=float32 batch=16
m2=4.57387 alpha=6.19872 lam4=0.5  dt=0.0005 steps=8000 burnin=2000 thin=20

[coarse step 400/8000] nsamp=0 wall=0.7s
[coarse step 800/8000] nsamp=0 wall=1.3s
[coarse step 1200/8000] nsamp=0 wall=1.9s
[coarse step 1600/8000] nsamp=0 wall=2.6s
[coarse step 2000/8000] nsamp=0 wall=3.2s
[coarse step 2400/8000] nsamp=20 wall=3.9s
[coarse step 2800/8000] nsamp=40 wall=4.6s
[coarse step 3200/8000] nsamp=60 wall=5.3s
[coarse step 3600/8000] nsamp=80 wall=5.9s
[coarse step 4000/8000] nsamp=100 wall=6.6s
[coarse step 4400/8000] nsamp=120 wall=7.3s
[coarse step 4800/8000] nsamp=140 wall=8.0s
[coarse step 5200/8000] nsamp=160 wall=8.7s
[coarse step 56

In [3]:
# ============================================================
# RG MATCHING LOOP (GPU): find coarse bare (m2, alpha) so that
# coarse effective low-k fit matches blocked target (m_eff2*, Z*).
#
# Input: unified_maxwell_phi4_drift_block_v2.npz (from your v2 run)
# Uses target: m_eff2_br, Z_br (blocked RG field, low-k fit)
#
# Model simulated on coarse lattice (Lc=L/2, spacing a'=2):
#   S = 1/2 m2 ||A||^2 + 1/2 alpha <A, curlcurl A> + 1/4 lam4 ||A||_4^4
# Langevin:
#   dA = -∇S dt + sqrt(2) dW
#
# Output: rg_match_quadratic.npz with iteration history and final params.
#
# This matches ONLY the quadratic sector (m_eff2, Z). If you want lam4 matching,
# we add a 3-parameter loop after this works.
# ============================================================

import math
import time
import argparse
import numpy as np
import torch
import torch.fft as fft


def set_seed(seed: int):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--inp", type=str, default="unified_v3_p4_stats.npz")

    p.add_argument("--device", type=str, default="cuda")
    p.add_argument("--dtype", type=str, default="float32", choices=["float32", "float64"])

    # coarse sim controls (per evaluation)
    p.add_argument("--steps", type=int, default=5000)
    p.add_argument("--burnin", type=int, default=1500)
    p.add_argument("--thin", type=int, default=20)
    p.add_argument("--dt", type=float, default=5e-4)
    p.add_argument("--batch", type=int, default=16)
    p.add_argument("--init_sigma", type=float, default=0.20)

    # fit controls
    p.add_argument("--lowk", type=int, default=256)

    # matching loop
    p.add_argument("--iters", type=int, default=6)
    p.add_argument("--damping", type=float, default=0.7)  # Newton step damping
    p.add_argument("--rel_step", type=float, default=0.08)  # finite-diff step size (relative)
    p.add_argument("--min_step", type=float, default=0.05)  # absolute minimum diff step

    # optional overrides for start
    p.add_argument("--m2_init", type=float, default=np.nan)
    p.add_argument("--alpha_init", type=float, default=np.nan)

    # optional override lam4 (otherwise uses lam4 from file)
    p.add_argument("--lam4", type=float, default=np.nan)

    p.add_argument("--out", type=str, default="rg_match_quadratic.npz")
    p.add_argument("--seed", type=int, default=0)

    args, _ = p.parse_known_args()
    return args


# ---------- FFT momentum with explicit spacing a ----------
def make_phat_1d(L: int, a: float, device: str, dtype: torch.dtype):
    f = torch.fft.fftfreq(L, d=1.0, device=device, dtype=dtype)
    return (2.0 / a) * torch.sin(math.pi * f)


def make_pgrids(ph: torch.Tensor):
    L = ph.numel()
    p0 = ph.view(L, 1, 1, 1)
    p1 = ph.view(1, L, 1, 1)
    p2 = ph.view(1, 1, L, 1)
    p3 = ph.view(1, 1, 1, L)
    return p0, p1, p2, p3


def project_perp_and_curlcurl(A: torch.Tensor, p0, p1, p2, p3, eps=1e-12):
    spatial = (-4, -3, -2, -1)
    Ahat = fft.fftn(A, dim=spatial, norm="ortho")  # complex

    p2grid = (p0 * p0 + p1 * p1 + p2 * p2 + p3 * p3)

    dot = (p0 * Ahat[:, 0] + p1 * Ahat[:, 1] + p2 * Ahat[:, 2] + p3 * Ahat[:, 3])

    inv = torch.zeros_like(p2grid)
    mask = p2grid > 0
    inv[mask] = 1.0 / (p2grid[mask] + eps)

    factor = dot * inv
    Aperp_hat = torch.stack(
        [
            Ahat[:, 0] - p0 * factor,
            Ahat[:, 1] - p1 * factor,
            Ahat[:, 2] - p2 * factor,
            Ahat[:, 3] - p3 * factor,
        ],
        dim=1,
    )

    curlcurl_hat = Aperp_hat * p2grid
    curlcurlA = fft.ifftn(curlcurl_hat, dim=spatial, norm="ortho").real
    return curlcurlA, Aperp_hat, p2grid


def pick_lowk(p2grid_torch, K):
    p2_flat = p2grid_torch.reshape(-1).detach().cpu().numpy()
    mask = p2_flat > 0
    idx_all = np.nonzero(mask)[0]
    idx_sort = idx_all[np.argsort(p2_flat[idx_all])]
    return p2_flat, idx_sort[: int(K)].astype(np.int64)


def lowk_fit_from_Pk(Pk_mean, p2_flat, lowk_idx):
    G = np.asarray(Pk_mean, dtype=np.float64)
    p2 = np.asarray(p2_flat[lowk_idx], dtype=np.float64)

    good = (G > 0) & np.isfinite(G) & np.isfinite(p2)
    G = G[good]
    p2 = p2[good]
    if G.size < 8:
        return float("nan"), float("nan"), int(G.size)

    y = 1.0 / G
    x = p2
    A = np.stack([np.ones_like(x), x], axis=1)
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    return float(coef[0]), float(coef[1]), int(G.size)


@torch.no_grad()
def eval_effective(Lc, batch, m2, alpha, lam4, dt, steps, burnin, thin, lowkK, device, dtype, seed):
    # Coarse lattice spacing corresponding to 2x block: a' = 2
    a = 2.0
    d = 4
    set_seed(seed)

    ph = make_phat_1d(Lc, a=a, device=device, dtype=dtype)
    p0, p1, p2, p3 = make_pgrids(ph)

    Afield = 0.20 * torch.randn((batch, d, Lc, Lc, Lc, Lc), device=device, dtype=dtype)

    _, _, p2grid = project_perp_and_curlcurl(Afield[:1], p0, p1, p2, p3)
    p2_flat, lowk_idx = pick_lowk(p2grid, lowkK)

    Pk_accum = np.zeros((len(lowk_idx),), dtype=np.float64)
    nsamp = 0

    for t in range(steps):
        curlcurlA, _, _ = project_perp_and_curlcurl(Afield, p0, p1, p2, p3)
        gradS = m2 * Afield + alpha * curlcurlA + lam4 * (Afield ** 3)
        Afield = Afield - dt * gradS + math.sqrt(2.0 * dt) * torch.randn_like(Afield)

        if t >= burnin and ((t - burnin) % thin == 0):
            curlcurlA, Aperp_hat, _ = project_perp_and_curlcurl(Afield, p0, p1, p2, p3)
            Ap = Aperp_hat.reshape(batch, d, -1)
            P = (Ap.real * Ap.real + Ap.imag * Ap.imag).mean(dim=(0, 1))
            Pk_accum += P[lowk_idx].detach().cpu().numpy()
            nsamp += 1

    if nsamp == 0:
        return dict(m_eff2=np.nan, Z=np.nan, nsamp=0, n_lowk=0)

    Pk_mean = Pk_accum / nsamp
    m_eff2, Z, n_lowk = lowk_fit_from_Pk(Pk_mean, p2_flat, lowk_idx)
    return dict(m_eff2=m_eff2, Z=Z, nsamp=nsamp, n_lowk=n_lowk)


def solve_2x2(J, f):
    # Solve J * dx = f for dx, with fallback
    J = np.asarray(J, dtype=np.float64).reshape(2, 2)
    f = np.asarray(f, dtype=np.float64).reshape(2)
    det = J[0,0]*J[1,1] - J[0,1]*J[1,0]
    if abs(det) < 1e-12 or not np.isfinite(det):
        return np.array([0.0, 0.0], dtype=np.float64)
    inv = (1.0/det) * np.array([[ J[1,1], -J[0,1]],
                                [-J[1,0],  J[0,0]]], dtype=np.float64)
    return inv @ f


def clamp_pos(x, lo=1e-6):
    if not np.isfinite(x):
        return lo
    return max(float(x), lo)


def main():
    args = parse_args()

    if args.device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA requested but not available")

    device = args.device
    dtype = torch.float32 if args.dtype == "float32" else torch.float64
    set_seed(args.seed)

    data = np.load(args.inp, allow_pickle=False)
    Lc = int(data["Lc"])
    target_m = float(data["m_eff2_br"])
    target_Z = float(data["Z_br"])
    lam4_file = float(data["lam4"]) if "lam4" in data else 0.5
    lam4 = lam4_file if np.isnan(args.lam4) else float(args.lam4)

    # init guesses (reasonable defaults: start near target, but not equal is fine)
    m2 = target_m if np.isnan(args.m2_init) else float(args.m2_init)
    alpha = target_Z if np.isnan(args.alpha_init) else float(args.alpha_init)
    m2 = clamp_pos(m2)
    alpha = clamp_pos(alpha)

    hist = []

    print("\n================= RG MATCH TARGET ================")
    print(f"Lc={Lc}  target (m_eff2, Z)=({target_m:.6g}, {target_Z:.6g})   lam4={lam4:.6g}")
    print("===================================================\n")

    for it in range(args.iters):
        t0 = time.time()

        # base eval
        base = eval_effective(
            Lc=Lc, batch=args.batch,
            m2=m2, alpha=alpha, lam4=lam4,
            dt=args.dt, steps=args.steps, burnin=args.burnin, thin=args.thin,
            lowkK=args.lowk, device=device, dtype=dtype,
            seed=args.seed + 1000*it + 0
        )
        f0 = np.array([base["m_eff2"] - target_m, base["Z"] - target_Z], dtype=np.float64)

        # finite differences
        dm = max(args.min_step, args.rel_step * max(1.0, m2))
        da = max(args.min_step, args.rel_step * max(1.0, alpha))

        e_m = eval_effective(
            Lc=Lc, batch=args.batch,
            m2=m2 + dm, alpha=alpha, lam4=lam4,
            dt=args.dt, steps=args.steps, burnin=args.burnin, thin=args.thin,
            lowkK=args.lowk, device=device, dtype=dtype,
            seed=args.seed + 1000*it + 1
        )
        e_a = eval_effective(
            Lc=Lc, batch=args.batch,
            m2=m2, alpha=alpha + da, lam4=lam4,
            dt=args.dt, steps=args.steps, burnin=args.burnin, thin=args.thin,
            lowkK=args.lowk, device=device, dtype=dtype,
            seed=args.seed + 1000*it + 2
        )

        f_m = np.array([e_m["m_eff2"] - target_m, e_m["Z"] - target_Z], dtype=np.float64)
        f_a = np.array([e_a["m_eff2"] - target_m, e_a["Z"] - target_Z], dtype=np.float64)

        # Jacobian columns approx: (f(m2+dm)-f0)/dm, (f(alpha+da)-f0)/da
        J = np.column_stack([(f_m - f0) / dm, (f_a - f0) / da])  # 2x2

        # Newton step: J dx = f0  => params := params - damping*dx
        dx = solve_2x2(J, f0)
        m2_new = clamp_pos(m2 - args.damping * dx[0])
        alpha_new = clamp_pos(alpha - args.damping * dx[1])

        wall = time.time() - t0

        row = dict(
            it=it,
            m2=m2, alpha=alpha,
            m_eff2=base["m_eff2"], Z=base["Z"],
            err_m=float(f0[0]), err_Z=float(f0[1]),
            dm=dm, da=da,
            J=J.copy(),
            dx=dx.copy(),
            m2_new=m2_new, alpha_new=alpha_new,
            wall=wall,
            nsamp=int(base["nsamp"]),
        )
        hist.append(row)

        print(f"[it {it}] m2={m2:.6g} alpha={alpha:.6g}  -> eff=({base['m_eff2']:.6g},{base['Z']:.6g}) "
              f"err=({f0[0]:+.3g},{f0[1]:+.3g})  step dx=({dx[0]:+.3g},{dx[1]:+.3g})  wall={wall:.1f}s")

        m2, alpha = m2_new, alpha_new

    # final evaluation at last params (fresh seed)
    final = eval_effective(
        Lc=Lc, batch=args.batch,
        m2=m2, alpha=alpha, lam4=lam4,
        dt=args.dt, steps=args.steps, burnin=args.burnin, thin=args.thin,
        lowkK=args.lowk, device=device, dtype=dtype,
        seed=args.seed + 999999
    )
    print("\n================= FINAL ================")
    print(f"m2_bare={m2:.6g} alpha_bare={alpha:.6g}")
    print(f"effective=({final['m_eff2']:.6g},{final['Z']:.6g})   target=({target_m:.6g},{target_Z:.6g})")
    print("=========================================\n")

    # pack history into arrays
    iters = len(hist)
    m2_hist = np.array([h["m2"] for h in hist], dtype=np.float64)
    a_hist = np.array([h["alpha"] for h in hist], dtype=np.float64)
    me_hist = np.array([h["m_eff2"] for h in hist], dtype=np.float64)
    Z_hist = np.array([h["Z"] for h in hist], dtype=np.float64)
    em_hist = np.array([h["err_m"] for h in hist], dtype=np.float64)
    eZ_hist = np.array([h["err_Z"] for h in hist], dtype=np.float64)
    wall_hist = np.array([h["wall"] for h in hist], dtype=np.float64)

    J_hist = np.stack([h["J"] for h in hist], axis=0) if iters > 0 else np.zeros((0,2,2), dtype=np.float64)
    dx_hist = np.stack([h["dx"] for h in hist], axis=0) if iters > 0 else np.zeros((0,2), dtype=np.float64)

    np.savez(
        args.out,
        Lc=Lc,
        target_m_eff2=target_m, target_Z=target_Z,
        lam4=lam4,
        # history
        m2_hist=m2_hist, alpha_hist=a_hist,
        m_eff2_hist=me_hist, Z_hist=Z_hist,
        err_m_hist=em_hist, err_Z_hist=eZ_hist,
        J_hist=J_hist, dx_hist=dx_hist,
        wall_hist=wall_hist,
        # final
        m2_final=m2, alpha_final=alpha,
        m_eff2_final=final["m_eff2"], Z_final=final["Z"],
        seed=args.seed,
        steps=args.steps, burnin=args.burnin, thin=args.thin, dt=args.dt, batch=args.batch,
        lowk=args.lowk, iters=args.iters, damping=args.damping, rel_step=args.rel_step, min_step=args.min_step,
    )
    print(f"[saved] {args.out}")


if __name__ == "__main__":
    main()



================= RG MATCH TARGET ================
Lc=16  target (m_eff2, Z)=(4.65554, 4.97629)   lam4=0.5

[it 0] m2=4.65554 alpha=4.97629  -> eff=(6.34046,6.8683) err=(+1.68,+1.89)  step dx=(+0.841,+2.45)  wall=24.8s
[it 1] m2=4.06684 alpha=3.26243  -> eff=(5.78469,4.11158) err=(+1.13,-0.865)  step dx=(+0.989,-0.556)  wall=24.8s
[it 2] m2=3.37445 alpha=3.65173  -> eff=(4.74498,4.93805) err=(+0.0894,-0.0382)  step dx=(+0.0722,-0.0613)  wall=24.8s
[it 3] m2=3.32394 alpha=3.69464  -> eff=(4.76784,4.58629) err=(+0.112,-0.39)  step dx=(+0.0491,-0.302)  wall=24.7s
[it 4] m2=3.28955 alpha=3.90579  -> eff=(4.70968,5.08234) err=(+0.0541,+0.106)  step dx=(+0.00344,+0.343)  wall=24.7s


KeyboardInterrupt: 

In [12]:
# ============================================================
# RG MATCH QUADRATIC (CRN + line search) — GPU (FIXED)
#
# Fixed: no torch.randn_like(..., generator=...) usage.
# ============================================================

import math
import time
import argparse
import numpy as np
import torch
import torch.fft as fft


def set_global_seed(seed: int):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--inp", type=str, default="unified_maxwell_phi4_drift_block_v2.npz")

    p.add_argument("--device", type=str, default="cuda")
    p.add_argument("--dtype", type=str, default="float32", choices=["float32", "float64"])

    # evaluation controls
    p.add_argument("--steps", type=int, default=6000)
    p.add_argument("--burnin", type=int, default=1800)
    p.add_argument("--thin", type=int, default=20)
    p.add_argument("--dt", type=float, default=5e-4)
    p.add_argument("--batch", type=int, default=16)
    p.add_argument("--init_sigma", type=float, default=0.20)
    p.add_argument("--lowk", type=int, default=256)

    # matching loop controls
    p.add_argument("--iters", type=int, default=10)
    p.add_argument("--tol", type=float, default=0.02, help="relative tolerance for BOTH m_eff2 and Z")
    p.add_argument("--rel_step", type=float, default=0.05, help="relative finite-diff step")
    p.add_argument("--min_step", type=float, default=0.02, help="absolute minimum finite-diff step")
    p.add_argument("--damping", type=float, default=1.0, help="initial Newton step scale")
    p.add_argument("--reg", type=float, default=1e-3, help="Tikhonov regularization for J^T J")
    p.add_argument("--max_backtrack", type=int, default=4, help="line search backtracks")
    p.add_argument("--pos_floor", type=float, default=1e-6, help="floor for positive params")

    # initial guesses (optional)
    p.add_argument("--m2_init", type=float, default=np.nan)
    p.add_argument("--alpha_init", type=float, default=np.nan)

    # lam4 choice
    p.add_argument("--lam4", type=float, default=np.nan)

    # CRN seeds
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--seed_init", type=int, default=12345)
    p.add_argument("--seed_noise", type=int, default=67890)

    p.add_argument("--out", type=str, default="rg_match_quadratic_crn.npz")

    args, _ = p.parse_known_args()
    return args


def make_phat_1d(L: int, a: float, device: str, dtype: torch.dtype):
    f = torch.fft.fftfreq(L, d=1.0, device=device, dtype=dtype)
    return (2.0 / a) * torch.sin(math.pi * f)


def make_pgrids(ph: torch.Tensor):
    L = ph.numel()
    p0 = ph.view(L, 1, 1, 1)
    p1 = ph.view(1, L, 1, 1)
    p2 = ph.view(1, 1, L, 1)
    p3 = ph.view(1, 1, 1, L)
    return p0, p1, p2, p3


def project_perp_and_curlcurl(A: torch.Tensor, p0, p1, p2, p3, eps=1e-12):
    spatial = (-4, -3, -2, -1)
    Ahat = fft.fftn(A, dim=spatial, norm="ortho")  # complex

    p2grid = (p0 * p0 + p1 * p1 + p2 * p2 + p3 * p3)
    dot = (p0 * Ahat[:, 0] + p1 * Ahat[:, 1] + p2 * Ahat[:, 2] + p3 * Ahat[:, 3])

    inv = torch.zeros_like(p2grid)
    mask = p2grid > 0
    inv[mask] = 1.0 / (p2grid[mask] + eps)

    factor = dot * inv
    Aperp_hat = torch.stack(
        [
            Ahat[:, 0] - p0 * factor,
            Ahat[:, 1] - p1 * factor,
            Ahat[:, 2] - p2 * factor,
            Ahat[:, 3] - p3 * factor,
        ],
        dim=1,
    )

    curlcurl_hat = Aperp_hat * p2grid
    curlcurlA = fft.ifftn(curlcurl_hat, dim=spatial, norm="ortho").real
    return curlcurlA, Aperp_hat, p2grid


def pick_lowk(p2grid_torch, K):
    p2_flat = p2grid_torch.reshape(-1).detach().cpu().numpy()
    mask = p2_flat > 0
    idx_all = np.nonzero(mask)[0]
    idx_sort = idx_all[np.argsort(p2_flat[idx_all])]
    return p2_flat, idx_sort[: int(K)].astype(np.int64)


def lowk_fit_from_Pk(Pk_mean, p2_flat, lowk_idx):
    G = np.asarray(Pk_mean, dtype=np.float64)
    p2 = np.asarray(p2_flat[lowk_idx], dtype=np.float64)

    good = (G > 0) & np.isfinite(G) & np.isfinite(p2)
    G = G[good]
    p2 = p2[good]
    if G.size < 8:
        return float("nan"), float("nan"), int(G.size)

    y = 1.0 / G
    x = p2
    A = np.stack([np.ones_like(x), x], axis=1)
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    return float(coef[0]), float(coef[1]), int(G.size)


def clamp_pos(x, lo):
    if not np.isfinite(x):
        return lo
    return max(float(x), lo)


@torch.no_grad()
def eval_effective_crn(
    Lc, batch, m2, alpha, lam4, dt, steps, burnin, thin, lowkK,
    device, dtype, init_sigma,
    seed_init, seed_noise
):
    a = 2.0
    d = 4

    ph = make_phat_1d(Lc, a=a, device=device, dtype=dtype)
    p0, p1, p2, p3 = make_pgrids(ph)

    gen_init = torch.Generator(device=device).manual_seed(int(seed_init))
    Afield = init_sigma * torch.randn(
        (batch, d, Lc, Lc, Lc, Lc),
        device=device, dtype=dtype, generator=gen_init
    )

    _, _, p2grid = project_perp_and_curlcurl(Afield[:1], p0, p1, p2, p3)
    p2_flat, lowk_idx = pick_lowk(p2grid, lowkK)

    gen_noise = torch.Generator(device=device).manual_seed(int(seed_noise))

    Pk_accum = np.zeros((len(lowk_idx),), dtype=np.float64)
    nsamp = 0
    sqrt_2dt = math.sqrt(2.0 * dt)

    for t in range(steps):
        curlcurlA, _, _ = project_perp_and_curlcurl(Afield, p0, p1, p2, p3)
        gradS = m2 * Afield + alpha * curlcurlA + lam4 * (Afield ** 3)

        noise = torch.randn(
            Afield.shape,
            device=device, dtype=dtype, generator=gen_noise
        )

        Afield = Afield - dt * gradS + sqrt_2dt * noise

        if t >= burnin and ((t - burnin) % thin == 0):
            _, Aperp_hat, _ = project_perp_and_curlcurl(Afield, p0, p1, p2, p3)
            Ap = Aperp_hat.reshape(batch, d, -1)
            P = (Ap.real * Ap.real + Ap.imag * Ap.imag).mean(dim=(0, 1))
            Pk_accum += P[lowk_idx].detach().cpu().numpy()
            nsamp += 1

    if nsamp == 0:
        return dict(m_eff2=np.nan, Z=np.nan, nsamp=0, n_lowk=0)

    Pk_mean = Pk_accum / nsamp
    m_eff2, Z, n_lowk = lowk_fit_from_Pk(Pk_mean, p2_flat, lowk_idx)
    return dict(m_eff2=m_eff2, Z=Z, nsamp=nsamp, n_lowk=n_lowk)


def newton_step_reg(J, f, reg):
    J = np.asarray(J, dtype=np.float64).reshape(2, 2)
    f = np.asarray(f, dtype=np.float64).reshape(2)
    JTJ = J.T @ J
    A = JTJ + float(reg) * np.eye(2, dtype=np.float64)
    b = J.T @ f
    try:
        dx = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        dx = np.zeros((2,), dtype=np.float64)
    return dx


def loss(f, target_m, target_Z):
    fm, fz = float(f[0]), float(f[1])
    return math.sqrt((fm / (abs(target_m) + 1e-12)) ** 2 + (fz / (abs(target_Z) + 1e-12)) ** 2)


def main():
    args = parse_args()

    if args.device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA requested but not available")

    device = args.device
    dtype = torch.float32 if args.dtype == "float32" else torch.float64
    set_global_seed(args.seed)

    data = np.load(args.inp, allow_pickle=False)
    Lc = int(data["L_block"])
    target_m = float(data["m_eff2_br"])
    target_Z = float(data["Z_br"])
    lam4_file = float(data["lam4"]) if "lam4" in data else 0.5
    lam4 = lam4_file if np.isnan(args.lam4) else float(args.lam4)

    m2 = target_m if np.isnan(args.m2_init) else float(args.m2_init)
    alpha = target_Z if np.isnan(args.alpha_init) else float(args.alpha_init)
    m2 = clamp_pos(m2, args.pos_floor)
    alpha = clamp_pos(alpha, args.pos_floor)

    print("\n================= RG MATCH TARGET (CRN) =================")
    print(f"Lc={Lc}  target (m_eff2, Z)=({target_m:.6g}, {target_Z:.6g})   lam4={lam4:.6g}")
    print(f"eval: steps={args.steps} burnin={args.burnin} thin={args.thin} dt={args.dt} batch={args.batch} lowk={args.lowk}")
    print("=========================================================\n")

    hist = []

    for it in range(args.iters):
        t0 = time.time()

        base = eval_effective_crn(
            Lc=Lc, batch=args.batch,
            m2=m2, alpha=alpha, lam4=lam4,
            dt=args.dt, steps=args.steps, burnin=args.burnin, thin=args.thin,
            lowkK=args.lowk, device=device, dtype=dtype, init_sigma=args.init_sigma,
            seed_init=args.seed_init, seed_noise=args.seed_noise
        )
        f0 = np.array([base["m_eff2"] - target_m, base["Z"] - target_Z], dtype=np.float64)
        L0 = loss(f0, target_m, target_Z)

        rel_m = abs(f0[0]) / (abs(target_m) + 1e-12)
        rel_Z = abs(f0[1]) / (abs(target_Z) + 1e-12)

        print(f"[it {it}] m2={m2:.6g} alpha={alpha:.6g}  eff=({base['m_eff2']:.6g},{base['Z']:.6g}) "
              f"relerr=({rel_m*100:.2f}%,{rel_Z*100:.2f}%)  nsamp={base['nsamp']}  loss={L0:.4g}")

        hist.append(dict(it=it, m2=m2, alpha=alpha, m_eff2=base["m_eff2"], Z=base["Z"], rel_m=rel_m, rel_Z=rel_Z, loss=L0))

        if rel_m < args.tol and rel_Z < args.tol:
            print("[stop] tolerance met.\n")
            break

        dm = max(args.min_step, args.rel_step * max(1.0, m2))
        da = max(args.min_step, args.rel_step * max(1.0, alpha))

        em = eval_effective_crn(
            Lc=Lc, batch=args.batch,
            m2=m2 + dm, alpha=alpha, lam4=lam4,
            dt=args.dt, steps=args.steps, burnin=args.burnin, thin=args.thin,
            lowkK=args.lowk, device=device, dtype=dtype, init_sigma=args.init_sigma,
            seed_init=args.seed_init, seed_noise=args.seed_noise
        )
        ea = eval_effective_crn(
            Lc=Lc, batch=args.batch,
            m2=m2, alpha=alpha + da, lam4=lam4,
            dt=args.dt, steps=args.steps, burnin=args.burnin, thin=args.thin,
            lowkK=args.lowk, device=device, dtype=dtype, init_sigma=args.init_sigma,
            seed_init=args.seed_init, seed_noise=args.seed_noise
        )

        fm = np.array([em["m_eff2"] - target_m, em["Z"] - target_Z], dtype=np.float64)
        fa = np.array([ea["m_eff2"] - target_m, ea["Z"] - target_Z], dtype=np.float64)

        J = np.column_stack([(fm - f0) / dm, (fa - f0) / da])  # 2x2
        dx = newton_step_reg(J, f0, args.reg)

        accepted = False
        step_scale = float(args.damping)

        for _ in range(args.max_backtrack + 1):
            m2_try = clamp_pos(m2 - step_scale * dx[0], args.pos_floor)
            a_try = clamp_pos(alpha - step_scale * dx[1], args.pos_floor)

            trial = eval_effective_crn(
                Lc=Lc, batch=args.batch,
                m2=m2_try, alpha=a_try, lam4=lam4,
                dt=args.dt, steps=args.steps, burnin=args.burnin, thin=args.thin,
                lowkK=args.lowk, device=device, dtype=dtype, init_sigma=args.init_sigma,
                seed_init=args.seed_init, seed_noise=args.seed_noise
            )

            f_try = np.array([trial["m_eff2"] - target_m, trial["Z"] - target_Z], dtype=np.float64)
            L_try = loss(f_try, target_m, target_Z)

            if np.isfinite(L_try) and (L_try < L0):
                accepted = True
                m2, alpha = m2_try, a_try
                break

            step_scale *= 0.5

        wall = time.time() - t0
        print(f"       step dx=({dx[0]:+.3g},{dx[1]:+.3g})  accepted={accepted}  wall={wall:.1f}s\n")

        if not accepted:
            print("[warn] no improving step found under backtracking.\n")

    final = eval_effective_crn(
        Lc=Lc, batch=args.batch,
        m2=m2, alpha=alpha, lam4=lam4,
        dt=args.dt, steps=args.steps, burnin=args.burnin, thin=args.thin,
        lowkK=args.lowk, device=device, dtype=dtype, init_sigma=args.init_sigma,
        seed_init=args.seed_init + 999, seed_noise=args.seed_noise + 999
    )

    f_fin = np.array([final["m_eff2"] - target_m, final["Z"] - target_Z], dtype=np.float64)
    rel_m_fin = abs(f_fin[0]) / (abs(target_m) + 1e-12)
    rel_Z_fin = abs(f_fin[1]) / (abs(target_Z) + 1e-12)

    print("================= FINAL (fresh noise) =================")
    print(f"m2_bare={m2:.6g}  alpha_bare={alpha:.6g}")
    print(f"effective=({final['m_eff2']:.6g},{final['Z']:.6g})  target=({target_m:.6g},{target_Z:.6g})")
    print(f"relerr=({rel_m_fin*100:.2f}%,{rel_Z_fin*100:.2f}%)  nsamp={final['nsamp']}")
    print("=======================================================\n")

    m2_hist = np.array([h["m2"] for h in hist], dtype=np.float64)
    a_hist = np.array([h["alpha"] for h in hist], dtype=np.float64)
    eff_m_hist = np.array([h["m_eff2"] for h in hist], dtype=np.float64)
    eff_Z_hist = np.array([h["Z"] for h in hist], dtype=np.float64)
    relm_hist = np.array([h["rel_m"] for h in hist], dtype=np.float64)
    relZ_hist = np.array([h["rel_Z"] for h in hist], dtype=np.float64)
    loss_hist = np.array([h["loss"] for h in hist], dtype=np.float64)

    np.savez(
        args.out,
        Lc=Lc,
        target_m_eff2=target_m,
        target_Z=target_Z,
        lam4=lam4,
        m2_hist=m2_hist,
        alpha_hist=a_hist,
        eff_m_eff2_hist=eff_m_hist,
        eff_Z_hist=eff_Z_hist,
        relerr_m_hist=relm_hist,
        relerr_Z_hist=relZ_hist,
        loss_hist=loss_hist,
        m2_final=m2,
        alpha_final=alpha,
        m_eff2_final=final["m_eff2"],
        Z_final=final["Z"],
        relerr_m_eff2=rel_m_fin,
        relerr_Z=rel_Z_fin,
        steps=args.steps, burnin=args.burnin, thin=args.thin, dt=args.dt, batch=args.batch, lowk=args.lowk,
        iters=args.iters, tol=args.tol, rel_step=args.rel_step, min_step=args.min_step,
        damping=args.damping, reg=args.reg, max_backtrack=args.max_backtrack, pos_floor=args.pos_floor,
        seed=args.seed, seed_init=args.seed_init, seed_noise=args.seed_noise,
    )
    print(f"[saved] {args.out}")


if __name__ == "__main__":
    main()



================= RG MATCH TARGET (CRN) =================
Lc=16  target (m_eff2, Z)=(4.57387, 6.19872)   lam4=0.5
eval: steps=6000 burnin=1800 thin=20 dt=0.0005 batch=16 lowk=256

[it 0] m2=4.57387 alpha=6.19872  eff=(6.27409,8.4626) relerr=(37.17%,36.52%)  nsamp=210  loss=0.5211
       step dx=(+1.32,+1.71)  accepted=True  wall=40.3s

[it 1] m2=3.25672 alpha=4.49353  eff=(4.59557,6.18341) relerr=(0.47%,0.25%)  nsamp=210  loss=0.005349
[stop] tolerance met.

================= FINAL (fresh noise) =================
m2_bare=3.25672  alpha_bare=4.49353
effective=(4.61189,6.14719)  target=(4.57387,6.19872)
relerr=(0.83%,0.83%)  nsamp=210

[saved] rg_match_quadratic_crn.npz


In [13]:
# ============================================================
# POST-MATCH CLOSURE RUN (GPU)
#
# Inputs:
#   - unified_maxwell_phi4_drift_block_v2.npz   (blocked target distributions + target low-k)
#   - rg_match_quadratic_crn.npz               (matched coarse bare params)
#
# Runs a coarse sim at Lc with (m2_bare, alpha_bare) from matcher,
# then compares:
#   (1) low-k (m_eff2, Z) vs blocked target
#   (2) V2 and LV2 quantiles vs blocked target
#   (3) one-sided drift fit for V2 vs blocked target drift summary
#
# Output:
#   - rg_postmatch_closure.npz
# ============================================================

import math
import time
import argparse
import numpy as np
import torch
import torch.fft as fft


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--target_npz", type=str, default="unified_maxwell_phi4_drift_block_v2.npz")
    p.add_argument("--match_npz", type=str, default="rg_match_quadratic_crn.npz")
    p.add_argument("--device", type=str, default="cuda")
    p.add_argument("--dtype", type=str, default="float32", choices=["float32", "float64"])

    # coarse sim
    p.add_argument("--steps", type=int, default=8000)
    p.add_argument("--burnin", type=int, default=2000)
    p.add_argument("--thin", type=int, default=20)
    p.add_argument("--dt", type=float, default=5e-4)
    p.add_argument("--batch", type=int, default=16)
    p.add_argument("--init_sigma", type=float, default=0.20)

    # fits
    p.add_argument("--lowk", type=int, default=256)
    p.add_argument("--qfit", type=float, default=0.995)
    p.add_argument("--lam_max", type=float, default=5.0)
    p.add_argument("--lam_grid", type=int, default=501)

    # RNG
    p.add_argument("--seed_init", type=int, default=12345)
    p.add_argument("--seed_noise", type=int, default=67890)

    p.add_argument("--out", type=str, default="rg_postmatch_closure.npz")

    args, _ = p.parse_known_args()
    return args


def make_phat_1d(L: int, a: float, device: str, dtype: torch.dtype):
    f = torch.fft.fftfreq(L, d=1.0, device=device, dtype=dtype)
    return (2.0 / a) * torch.sin(math.pi * f)


def make_pgrids(ph: torch.Tensor):
    L = ph.numel()
    p0 = ph.view(L, 1, 1, 1)
    p1 = ph.view(1, L, 1, 1)
    p2 = ph.view(1, 1, L, 1)
    p3 = ph.view(1, 1, 1, L)
    return p0, p1, p2, p3


def project_perp_and_curlcurl(A: torch.Tensor, p0, p1, p2, p3, eps=1e-12):
    spatial = (-4, -3, -2, -1)
    Ahat = fft.fftn(A, dim=spatial, norm="ortho")  # complex
    p2grid = (p0 * p0 + p1 * p1 + p2 * p2 + p3 * p3)

    dot = (p0 * Ahat[:, 0] + p1 * Ahat[:, 1] + p2 * Ahat[:, 2] + p3 * Ahat[:, 3])

    inv = torch.zeros_like(p2grid)
    mask = p2grid > 0
    inv[mask] = 1.0 / (p2grid[mask] + eps)

    factor = dot * inv
    Aperp_hat = torch.stack(
        [
            Ahat[:, 0] - p0 * factor,
            Ahat[:, 1] - p1 * factor,
            Ahat[:, 2] - p2 * factor,
            Ahat[:, 3] - p3 * factor,
        ],
        dim=1,
    )

    curlcurl_hat = Aperp_hat * p2grid
    curlcurlA = fft.ifftn(curlcurl_hat, dim=spatial, norm="ortho").real
    return curlcurlA, Aperp_hat, p2grid


def pick_lowk(p2grid_torch, K):
    p2_flat = p2grid_torch.reshape(-1).detach().cpu().numpy()
    mask = p2_flat > 0
    idx_all = np.nonzero(mask)[0]
    idx_sort = idx_all[np.argsort(p2_flat[idx_all])]
    return p2_flat, idx_sort[: int(K)].astype(np.int64)


def lowk_mass_fit(Pk_mean, p2_flat, lowk_idx):
    G = np.asarray(Pk_mean, dtype=np.float64)
    p2 = np.asarray(p2_flat[lowk_idx], dtype=np.float64)

    good = (G > 0) & np.isfinite(G) & np.isfinite(p2)
    G = G[good]
    p2 = p2[good]
    if G.size < 8:
        return dict(m_eff2=np.nan, Z=np.nan, n=int(G.size))

    y = 1.0 / G
    x = p2
    A = np.stack([np.ones_like(x), x], axis=1)
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    return dict(m_eff2=float(coef[0]), Z=float(coef[1]), n=int(G.size))


def one_sided_drift_fit(V_np, LV_np, lam_max=5.0, lam_grid=501, q=0.995):
    V = np.asarray(V_np, dtype=np.float64)
    LV = np.asarray(LV_np, dtype=np.float64)

    lam_vals = np.linspace(0.0, float(lam_max), int(lam_grid))
    b_vals = np.empty_like(lam_vals)
    viol_vals = np.empty_like(lam_vals)
    score_vals = np.empty_like(lam_vals)

    eps = 1e-12
    for i, lam in enumerate(lam_vals):
        z = LV + lam * V
        b = np.quantile(z, q)
        b_vals[i] = b
        viol_vals[i] = np.mean(LV > (-lam * V + b))
        score_vals[i] = lam / (b + eps)

    i_star = int(np.argmax(score_vals))
    return dict(
        lam_star=float(lam_vals[i_star]),
        b_star=float(b_vals[i_star]),
        score_star=float(score_vals[i_star]),
        lam_grid=lam_vals,
        b_grid=b_vals,
        viol_grid=viol_vals,
        score_grid=score_vals,
    )


def qstats(x):
    x = np.asarray(x, dtype=np.float64)
    if x.size == 0:
        return dict(mean=np.nan, q50=np.nan, q90=np.nan, q99=np.nan)
    return dict(
        mean=float(x.mean()),
        q50=float(np.quantile(x, 0.5)),
        q90=float(np.quantile(x, 0.9)),
        q99=float(np.quantile(x, 0.99)),
    )


@torch.no_grad()
def run_coarse_sim(
    Lc, batch, m2, alpha, lam4,
    dt, steps, burnin, thin,
    lowkK, qfit, lam_max, lam_grid,
    device, dtype,
    init_sigma, seed_init, seed_noise
):
    d = 4
    a = 2.0

    ph = make_phat_1d(Lc, a=a, device=device, dtype=dtype)
    p0, p1, p2, p3 = make_pgrids(ph)

    gen_init = torch.Generator(device=device).manual_seed(int(seed_init))
    Afield = init_sigma * torch.randn((batch, d, Lc, Lc, Lc, Lc), device=device, dtype=dtype, generator=gen_init)

    _, _, p2grid = project_perp_and_curlcurl(Afield[:1], p0, p1, p2, p3)
    p2_flat, lowk_idx = pick_lowk(p2grid, lowkK)

    gen_noise = torch.Generator(device=device).manual_seed(int(seed_noise))
    sqrt_2dt = math.sqrt(2.0 * dt)
    n = float(d * (Lc ** 4))

    V2_list, LV2_list = [], []
    Pk_accum = np.zeros((len(lowk_idx),), dtype=np.float64)
    nsamp = 0
    t0 = time.time()

    for t in range(steps):
        curlcurlA, _, _ = project_perp_and_curlcurl(Afield, p0, p1, p2, p3)
        gradS = m2 * Afield + alpha * curlcurlA + lam4 * (Afield ** 3)
        noise = torch.randn(Afield.shape, device=device, dtype=dtype, generator=gen_noise)
        Afield = Afield - dt * gradS + sqrt_2dt * noise

        if t >= burnin and ((t - burnin) % thin == 0):
            curlcurlA, Aperp_hat, _ = project_perp_and_curlcurl(Afield, p0, p1, p2, p3)
            gradS = m2 * Afield + alpha * curlcurlA + lam4 * (Afield ** 3)

            V2 = (Afield * Afield).flatten(1).sum(-1) / n
            inner = (gradS * Afield).flatten(1).sum(-1)
            LV2 = 2.0 - (2.0 / n) * inner

            V2_list.append(V2.detach().cpu().numpy())
            LV2_list.append(LV2.detach().cpu().numpy())

            Ap = Aperp_hat.reshape(batch, d, -1)
            P = (Ap.real * Ap.real + Ap.imag * Ap.imag).mean(dim=(0, 1))
            Pk_accum += P[lowk_idx].detach().cpu().numpy()

            nsamp += 1

        if (t + 1) % max(400, thin * 20) == 0:
            print(f"[coarse step {t+1}/{steps}] nsamp={nsamp} wall={time.time()-t0:.1f}s")

    V2_np = np.concatenate(V2_list, axis=0) if V2_list else np.empty((0,), dtype=np.float64)
    LV2_np = np.concatenate(LV2_list, axis=0) if LV2_list else np.empty((0,), dtype=np.float64)

    Pk_mean = (Pk_accum / nsamp) if nsamp > 0 else Pk_accum * np.nan
    mf = lowk_mass_fit(Pk_mean, p2_flat, lowk_idx)
    drift = one_sided_drift_fit(V2_np, LV2_np, lam_max=lam_max, lam_grid=lam_grid, q=qfit)

    return dict(
        V2=V2_np, LV2=LV2_np,
        Pk_mean=Pk_mean, p2_flat=p2_flat, lowk_idx=lowk_idx,
        m_eff2=mf["m_eff2"], Z=mf["Z"], n_lowk=mf["n"],
        drift=drift,
        nsamp=nsamp,
    )


def relerr(a, b):
    return float(abs(a - b) / (abs(b) + 1e-12))


def main():
    args = parse_args()

    if args.device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA requested but not available")

    device = args.device
    dtype = torch.float32 if args.dtype == "float32" else torch.float64

    targ = np.load(args.target_npz, allow_pickle=False)
    match = np.load(args.match_npz, allow_pickle=False)

    Lc = int(targ["L_block"])
    lam4 = float(targ["lam4"]) if "lam4" in targ else 0.5

    # Blocked RG targets (from v2 run)
    target_m = float(targ["m_eff2_br"])
    target_Z = float(targ["Z_br"])
    V2_t = targ["V2_block_rg"]
    LV2_t = targ["LV2_block_rg"]

    # Matched coarse bare params
    m2_bare = float(match["m2_final"])
    alpha_bare = float(match["alpha_final"])

    print("\n================= POST-MATCH CLOSURE =================")
    print(f"Lc={Lc}  lam4={lam4}")
    print(f"target (m_eff2,Z)=({target_m:.6g},{target_Z:.6g})")
    print(f"matched bare (m2,alpha)=({m2_bare:.6g},{alpha_bare:.6g})")
    print("======================================================\n")

    targ_V2 = qstats(V2_t)
    targ_LV2 = qstats(LV2_t)

    out = run_coarse_sim(
        Lc=Lc, batch=args.batch,
        m2=m2_bare, alpha=alpha_bare, lam4=lam4,
        dt=args.dt, steps=args.steps, burnin=args.burnin, thin=args.thin,
        lowkK=args.lowk, qfit=args.qfit, lam_max=args.lam_max, lam_grid=args.lam_grid,
        device=device, dtype=dtype,
        init_sigma=args.init_sigma, seed_init=args.seed_init, seed_noise=args.seed_noise,
    )

    sim_V2 = qstats(out["V2"])
    sim_LV2 = qstats(out["LV2"])

    e_m = relerr(out["m_eff2"], target_m)
    e_Z = relerr(out["Z"], target_Z)
    e_V2 = relerr(sim_V2["mean"], targ_V2["mean"])
    e_LV2 = relerr(sim_LV2["mean"], targ_LV2["mean"])

    print("\n================= TARGET vs SIM =================")
    print(f"[low-k]   target m_eff2={target_m:.6g}  sim m_eff2={out['m_eff2']:.6g}  relerr={e_m*100:.2f}%")
    print(f"[low-k]   target Z     ={target_Z:.6g}  sim Z     ={out['Z']:.6g}  relerr={e_Z*100:.2f}%")
    print("")
    print(f"[V2 mean] target={targ_V2['mean']:.6g}  sim={sim_V2['mean']:.6g}  relerr={e_V2*100:.2f}%")
    print(f"[LV2 mean]target={targ_LV2['mean']:.6g}  sim={sim_LV2['mean']:.6g}  relerr={e_LV2*100:.2f}%")
    print("")
    print(f"[target V2]  mean={targ_V2['mean']:.6g} q50={targ_V2['q50']:.6g} q90={targ_V2['q90']:.6g} q99={targ_V2['q99']:.6g}")
    print(f"[sim V2]     mean={sim_V2['mean']:.6g} q50={sim_V2['q50']:.6g} q90={sim_V2['q90']:.6g} q99={sim_V2['q99']:.6g}")
    print(f"[target LV2] mean={targ_LV2['mean']:.6g} q50={targ_LV2['q50']:.6g} q90={targ_LV2['q90']:.6g} q99={targ_LV2['q99']:.6g}")
    print(f"[sim LV2]    mean={sim_LV2['mean']:.6g} q50={sim_LV2['q50']:.6g} q90={sim_LV2['q90']:.6g} q99={sim_LV2['q99']:.6g}")
    print("")
    print(f"[sim drift fit] lam*={out['drift']['lam_star']:.6g}  b*={out['drift']['b_star']:.6g}  viol@*={out['drift']['viol_grid'][np.argmax(out['drift']['score_grid'])]*100:.3f}%")
    print("=============================================\n")

    np.savez(
        args.out,
        Lc=Lc, lam4=lam4,
        target_m_eff2=target_m, target_Z=target_Z,
        target_V2=V2_t, target_LV2=LV2_t,
        m2_bare=m2_bare, alpha_bare=alpha_bare,
        sim_m_eff2=out["m_eff2"], sim_Z=out["Z"],
        sim_V2=out["V2"], sim_LV2=out["LV2"],
        sim_Pk_mean=out["Pk_mean"], sim_p2_flat=out["p2_flat"], sim_lowk_idx=out["lowk_idx"],
        drift_lam_star=out["drift"]["lam_star"], drift_b_star=out["drift"]["b_star"], drift_score_star=out["drift"]["score_star"],
        relerr_m_eff2=e_m, relerr_Z=e_Z, relerr_V2_mean=e_V2, relerr_LV2_mean=e_LV2,
        steps=args.steps, burnin=args.burnin, thin=args.thin, dt=args.dt, batch=args.batch,
        lowk=args.lowk, qfit=args.qfit, lam_max=args.lam_max, lam_grid=args.lam_grid,
        seed_init=args.seed_init, seed_noise=args.seed_noise,
    )
    print(f"[saved] {args.out}")


if __name__ == "__main__":
    main()



================= POST-MATCH CLOSURE =================
Lc=16  lam4=0.5
target (m_eff2,Z)=(4.57387,6.19872)
matched bare (m2,alpha)=(3.25672,4.49353)

[coarse step 400/8000] nsamp=0 wall=0.7s
[coarse step 800/8000] nsamp=0 wall=1.3s
[coarse step 1200/8000] nsamp=0 wall=2.0s
[coarse step 1600/8000] nsamp=0 wall=2.6s
[coarse step 2000/8000] nsamp=0 wall=3.3s
[coarse step 2400/8000] nsamp=20 wall=3.9s
[coarse step 2800/8000] nsamp=40 wall=4.6s
[coarse step 3200/8000] nsamp=60 wall=5.3s
[coarse step 3600/8000] nsamp=80 wall=6.0s
[coarse step 4000/8000] nsamp=100 wall=6.7s
[coarse step 4400/8000] nsamp=120 wall=7.4s
[coarse step 4800/8000] nsamp=140 wall=8.1s
[coarse step 5200/8000] nsamp=160 wall=8.8s
[coarse step 5600/8000] nsamp=180 wall=9.5s
[coarse step 6000/8000] nsamp=200 wall=10.1s
[coarse step 6400/8000] nsamp=220 wall=10.8s
[coarse step 6800/8000] nsamp=240 wall=11.5s
[coarse step 7200/8000] nsamp=260 wall=12.2s
[coarse step 7600/8000] nsamp=280 wall=12.9s
[coarse step 8000/8000] 

In [15]:
import numpy as np

def qstats(x, name):
    x = np.asarray(x, dtype=np.float64)
    print(f"{name}: mean={x.mean():.6g}  q50={np.quantile(x,0.5):.6g}  q90={np.quantile(x,0.9):.6g}  q99={np.quantile(x,0.99):.6g}")

def drift_fit_quantile(V2, LV2, lam_max=5.0, lam_grid=501, q=0.995):
    V2 = np.asarray(V2, dtype=np.float64)
    LV2 = np.asarray(LV2, dtype=np.float64)
    lam = np.linspace(0.0, float(lam_max), int(lam_grid))
    b = np.empty_like(lam)
    viol = np.empty_like(lam)
    score = np.empty_like(lam)
    eps = 1e-12
    for i, l in enumerate(lam):
        z = LV2 + l * V2
        b[i] = np.quantile(z, q)
        viol[i] = np.mean(LV2 > (-l * V2 + b[i]))
        score[i] = l / (b[i] + eps)
    i0 = int(np.argmax(score))
    return dict(lam_star=float(lam[i0]), b_star=float(b[i0]), viol=float(viol[i0]), score=float(score[i0]))

# -------------------------
# Load files
# -------------------------
v2 = np.load("unified_maxwell_phi4_drift_block_v2.npz", allow_pickle=False)
match = np.load("rg_match_quadratic_crn.npz", allow_pickle=False)

# Fine couplings used when v2 computed LV2_block_rg
m2_f = float(v2["m2"])
alpha_f = float(v2["alpha"])
lam4_f = float(v2["lam4"])

Lc = int(v2["L_block"])
d = int(v2["d"])
n = float(d * (Lc ** 4))

# Blocked RG arrays from v2
S = np.asarray(v2["S_block_rg"], dtype=np.float64)         # energy under (m2_f, alpha_f, lam4_f)
V2 = np.asarray(v2["V2_block_rg"], dtype=np.float64)       # = N2 / n
LV2_saved = np.asarray(v2["LV2_block_rg"], dtype=np.float64)

# Matched coarse bare couplings (for quadratic sector)
m2_b = float(match["m2_final"])
alpha_b = float(match["alpha_final"])
lam4_b = lam4_f  # keep same for now; we'll also solve for lam4 that forces mean drift

print(f"[meta] Lc={Lc} d={d} n={n:.0f}")
print(f"[fine couplings]  m2={m2_f} alpha={alpha_f} lam4={lam4_f}")
print(f"[matched bare]    m2={m2_b} alpha={alpha_b} lam4={lam4_b}\n")

# -------------------------
# Reconstruct sufficient stats per sample:
# N2 = sum A^2, C = <A, curlcurl A>, N4 = sum A^4
#
# We have:
#   S = 0.5 m2 N2 + 0.5 alpha C + 0.25 lam4 N4
#   inner = <∇S, A> = m2 N2 + alpha C + lam4 N4
# and LV2 = 2 - (2/n) inner  => inner = 0.5 n (2 - LV2)
# Solve for N4 and C using known (m2_f, alpha_f, lam4_f) and known N2,S,LV2.
# -------------------------
N2 = V2 * n
inner = 0.5 * n * (2.0 - LV2_saved)

# N4 = 2*(inner - 2S)/lam4
# (derived from: (lam4/2)N4 = inner - 2S)
if lam4_f == 0.0:
    raise RuntimeError("lam4_f=0, cannot reconstruct N4 from (S,LV2).")
N4 = 2.0 * (inner - 2.0 * S) / lam4_f

# C from: alpha C = inner - m2 N2 - lam4 N4
if alpha_f == 0.0:
    raise RuntimeError("alpha_f=0, cannot reconstruct C.")
C = (inner - m2_f * N2 - lam4_f * N4) / alpha_f

# sanity: recompute LV2 under fine couplings and compare to saved
inner_check = m2_f * N2 + alpha_f * C + lam4_f * N4
LV2_check = 2.0 - (2.0 / n) * inner_check
max_abs = np.max(np.abs(LV2_check - LV2_saved))
print(f"[sanity] max|LV2_check - LV2_saved| = {max_abs:.3e}  (should be ~0)\n")

# -------------------------
# Recompute LV2 on blocked samples under matched bare couplings
# -------------------------
inner_b = m2_b * N2 + alpha_b * C + lam4_b * N4
LV2_b = 2.0 - (2.0 / n) * inner_b

qstats(V2, "blocked V2")
qstats(LV2_saved, "blocked LV2 (under fine couplings)")
qstats(LV2_b, "blocked LV2 (under matched bare couplings)")

# Stationarity diagnostic: mean(LV2)=0 <=> mean(inner)=n
print("\n[stationarity check on blocked samples under matched bare]")
print(f"mean(inner_b)={inner_b.mean():.6g}  target n={n:.6g}  relerr={(abs(inner_b.mean()-n)/(abs(n)+1e-12))*100:.3f}%")

# Solve lam4 that forces mean(LV2)=0 given (m2_b, alpha_b):
# mean(inner) = m2_b mean(N2) + alpha_b mean(C) + lam4* mean(N4) = n
lam4_star = (n - (m2_b * N2.mean() + alpha_b * C.mean())) / (N4.mean() + 1e-12)
print(f"lam4_star_for_mean_drift0 = {lam4_star:.6g}")

inner_b_star = m2_b * N2 + alpha_b * C + lam4_star * N4
LV2_b_star = 2.0 - (2.0 / n) * inner_b_star
qstats(LV2_b_star, "blocked LV2 (matched m2,alpha; lam4 tuned for mean drift 0)")

# Optional: drift fit numbers on blocked samples under matched bare couplings
fit_b = drift_fit_quantile(V2, LV2_b, lam_max=5.0, lam_grid=501, q=0.995)
fit_bstar = drift_fit_quantile(V2, LV2_b_star, lam_max=5.0, lam_grid=501, q=0.995)
print("\n[drift-fit on blocked samples]")
print(f"matched bare: lam*={fit_b['lam_star']:.6g} b*={fit_b['b_star']:.6g} viol={fit_b['viol']*100:.3f}% score={fit_b['score']:.3g}")
print(f"lam4 tuned : lam*={fit_bstar['lam_star']:.6g} b*={fit_bstar['b_star']:.6g} viol={fit_bstar['viol']*100:.3f}% score={fit_bstar['score']:.3g}")

# Save reconstructed stats and recomputed LV2 variants
np.savez(
    "blocked_recompute_LV2_stats.npz",
    Lc=Lc, d=d, n=n,
    m2_f=m2_f, alpha_f=alpha_f, lam4_f=lam4_f,
    m2_b=m2_b, alpha_b=alpha_b, lam4_b=lam4_b, lam4_star=lam4_star,
    S_block=S, V2_block=V2, LV2_saved=LV2_saved,
    N2=N2, C=C, N4=N4,
    LV2_b=LV2_b, LV2_b_star=LV2_b_star,
)
print("\n[saved] blocked_recompute_LV2_stats.npz")


[meta] Lc=16 d=4 n=262144
[fine couplings]  m2=0.3 alpha=1.0 lam4=0.5
[matched bare]    m2=3.2567246369102083 alpha=4.4935321292574475 lam4=0.5

[sanity] max|LV2_check - LV2_saved| = 0.000e+00  (should be ~0)

blocked V2: mean=0.121648  q50=0.123716  q90=0.124965  q99=0.125402
blocked LV2 (under fine couplings): mean=1.58524  q50=1.5788  q90=1.6087  q99=1.62836
blocked LV2 (under matched bare couplings): mean=-0.174278  q50=-0.206182  q90=-0.0592146  q99=0.0378703

[stationarity check on blocked samples under matched bare]
mean(inner_b)=284987  target n=262144  relerr=8.714%
lam4_star_for_mean_drift0 = -1.47913
blocked LV2 (matched m2,alpha; lam4 tuned for mean drift 0): mean=-2.72375e-16  q50=-0.0257909  q90=0.0951583  q99=0.176602

[drift-fit on blocked samples]
matched bare: lam*=5 b*=0.586726 viol=0.500% score=8.52
lam4 tuned : lam*=5 b*=0.724038 viol=0.500% score=6.91

[saved] blocked_recompute_LV2_stats.npz


In [2]:
# ============================================================
# UNIFIED RUN v3 (GPU): collect operator stats + p^4 quadratic fit
#
# Outputs (fine, block-naive, block-RG):
#   per-sample: N2, C1=<A,curlcurlA>, C2=<curlcurlA,curlcurlA>, N4
#   low-k fit: 1/G(p) ≈ m_eff2 + Z p2 + gamma p4
#
# This gives a real new result: whether blocking generates a p^4 operator.
# ============================================================

import math
import time
import argparse
import numpy as np
import torch
import torch.fft as fft


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--L", type=int, default=32)
    p.add_argument("--batch", type=int, default=8)

    p.add_argument("--m2", type=float, default=0.30)
    p.add_argument("--alpha", type=float, default=1.0)
    p.add_argument("--lam4", type=float, default=0.50)

    p.add_argument("--dt", type=float, default=5e-4)
    p.add_argument("--steps", type=int, default=6000)
    p.add_argument("--burnin", type=int, default=1500)
    p.add_argument("--thin", type=int, default=20)

    p.add_argument("--init_sigma", type=float, default=0.20)
    p.add_argument("--rg_field_scale", type=float, default=2.0)

    p.add_argument("--lowk", type=int, default=256)

    p.add_argument("--device", type=str, default="cuda")
    p.add_argument("--dtype", type=str, default="float32", choices=["float32", "float64"])
    p.add_argument("--seed", type=int, default=0)

    p.add_argument("--out", type=str, default="unified_v3_p4_stats.npz")

    args, _ = p.parse_known_args()
    return args


def set_seed(seed: int):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)


def make_phat_1d(L: int, a: float, device: str, dtype: torch.dtype):
    f = torch.fft.fftfreq(L, d=1.0, device=device, dtype=dtype)
    return (2.0 / a) * torch.sin(math.pi * f)


def make_pgrids(ph: torch.Tensor):
    L = ph.numel()
    p0 = ph.view(L, 1, 1, 1)
    p1 = ph.view(1, L, 1, 1)
    p2 = ph.view(1, 1, L, 1)
    p3 = ph.view(1, 1, 1, L)
    return p0, p1, p2, p3


def project_perp_and_curlcurl(A: torch.Tensor, p0, p1, p2, p3, eps=1e-12):
    spatial = (-4, -3, -2, -1)
    Ahat = fft.fftn(A, dim=spatial, norm="ortho")  # complex

    p2grid = (p0 * p0 + p1 * p1 + p2 * p2 + p3 * p3)

    dot = (p0 * Ahat[:, 0] + p1 * Ahat[:, 1] + p2 * Ahat[:, 2] + p3 * Ahat[:, 3])

    inv = torch.zeros_like(p2grid)
    mask = p2grid > 0
    inv[mask] = 1.0 / (p2grid[mask] + eps)

    factor = dot * inv
    Aperp_hat = torch.stack(
        [
            Ahat[:, 0] - p0 * factor,
            Ahat[:, 1] - p1 * factor,
            Ahat[:, 2] - p2 * factor,
            Ahat[:, 3] - p3 * factor,
        ],
        dim=1,
    )

    curlcurl_hat = Aperp_hat * p2grid
    curlcurlA = fft.ifftn(curlcurl_hat, dim=spatial, norm="ortho").real
    return curlcurlA, Aperp_hat, p2grid


def block2_avg(A):
    B, d, L, _, _, _ = A.shape
    assert L % 2 == 0
    L2 = L // 2
    return A.reshape(B, d, L2, 2, L2, 2, L2, 2, L2, 2).mean(dim=(3, 5, 7, 9))


def pick_lowk(p2grid_torch, K):
    p2_flat = p2grid_torch.reshape(-1).detach().cpu().numpy()
    mask = p2_flat > 0
    idx_all = np.nonzero(mask)[0]
    idx_sort = idx_all[np.argsort(p2_flat[idx_all])]
    lowk_idx = idx_sort[: int(K)].astype(np.int64)
    return p2_flat, lowk_idx


def fit_invprop_p2_p4(Pk_mean, p2_flat, lowk_idx):
    # Fit: 1/G = a + b p2 + c p4  on selected low-k indices
    G = np.asarray(Pk_mean, dtype=np.float64)
    p2 = np.asarray(p2_flat[lowk_idx], dtype=np.float64)
    p4 = p2 * p2

    good = (G > 0) & np.isfinite(G) & np.isfinite(p2)
    G = G[good]
    p2 = p2[good]
    p4 = p4[good]
    if G.size < 12:
        return dict(m_eff2=np.nan, Z=np.nan, gamma=np.nan, n=int(G.size))

    y = 1.0 / G
    X = np.stack([np.ones_like(p2), p2, p4], axis=1)
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    return dict(m_eff2=float(coef[0]), Z=float(coef[1]), gamma=float(coef[2]), n=int(G.size))


def main():
    args = parse_args()

    if args.device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA requested but not available")

    device = args.device
    dtype = torch.float32 if args.dtype == "float32" else torch.float64
    set_seed(args.seed)

    L = args.L
    d = 4
    B = args.batch
    assert L % 2 == 0

    # fine grids (a=1), coarse grids (a'=2)
    ph = make_phat_1d(L, a=1.0, device=device, dtype=dtype)
    p0, p1, p2, p3 = make_pgrids(ph)

    Lc = L // 2
    ph_c = make_phat_1d(Lc, a=2.0, device=device, dtype=dtype)
    q0, q1, q2, q3 = make_pgrids(ph_c)

    # init
    A = args.init_sigma * torch.randn((B, d, L, L, L, L), device=device, dtype=dtype)

    # low-k indices (fine and coarse)
    with torch.no_grad():
        _, _, p2grid = project_perp_and_curlcurl(A[:1], p0, p1, p2, p3)
        p2_flat, lowk_idx = pick_lowk(p2grid, args.lowk)

        Ab0 = block2_avg(A[:1])
        _, _, p2grid_c = project_perp_and_curlcurl(Ab0, q0, q1, q2, q3)
        p2_flat_c, lowk_idx_c = pick_lowk(p2grid_c, args.lowk)

    # accumulators: per-sample stats
    N2_f, C1_f, C2_f, N4_f = [], [], [], []
    N2_bn, C1_bn, C2_bn, N4_bn = [], [], [], []
    N2_br, C1_br, C2_br, N4_br = [], [], [], []

    # spectra accumulators
    Pk_f = np.zeros((len(lowk_idx),), dtype=np.float64)
    Pk_bn = np.zeros((len(lowk_idx_c),), dtype=np.float64)
    Pk_br = np.zeros((len(lowk_idx_c),), dtype=np.float64)

    nsamp = 0
    t0 = time.time()
    sqrt_2dt = math.sqrt(2.0 * args.dt)

    for t in range(args.steps):
        # evolve fine field
        curlcurlA, _, _ = project_perp_and_curlcurl(A, p0, p1, p2, p3)
        gradS = args.m2 * A + args.alpha * curlcurlA + args.lam4 * (A ** 3)
        A = A - args.dt * gradS + sqrt_2dt * torch.randn_like(A)

        # sample
        if t >= args.burnin and ((t - args.burnin) % args.thin == 0):
            with torch.no_grad():
                # ---- fine stats ----
                curlcurlA, Aperp_hat, _ = project_perp_and_curlcurl(A, p0, p1, p2, p3)
                N2 = (A * A).flatten(1).sum(-1)
                C1 = (A * curlcurlA).flatten(1).sum(-1)
                C2 = (curlcurlA * curlcurlA).flatten(1).sum(-1)
                N4 = (A ** 4).flatten(1).sum(-1)

                N2_f.append(N2.cpu().numpy())
                C1_f.append(C1.cpu().numpy())
                C2_f.append(C2.cpu().numpy())
                N4_f.append(N4.cpu().numpy())

                Ap = Aperp_hat.reshape(B, d, -1)
                P = (Ap.real * Ap.real + Ap.imag * Ap.imag).mean(dim=(0, 1))
                Pk_f += P[lowk_idx].cpu().numpy()

                # ---- block naive ----
                Abn = block2_avg(A)
                curlbn, Aperp_bn, _ = project_perp_and_curlcurl(Abn, q0, q1, q2, q3)
                N2n = (Abn * Abn).flatten(1).sum(-1)
                C1n = (Abn * curlbn).flatten(1).sum(-1)
                C2n = (curlbn * curlbn).flatten(1).sum(-1)
                N4n = (Abn ** 4).flatten(1).sum(-1)

                N2_bn.append(N2n.cpu().numpy())
                C1_bn.append(C1n.cpu().numpy())
                C2_bn.append(C2n.cpu().numpy())
                N4_bn.append(N4n.cpu().numpy())

                Apn = Aperp_bn.reshape(B, d, -1)
                Pn = (Apn.real * Apn.real + Apn.imag * Apn.imag).mean(dim=(0, 1))
                Pk_bn += Pn[lowk_idx_c].cpu().numpy()

                # ---- block RG (field rescale) ----
                Abr = args.rg_field_scale * Abn
                curlbr, Aperp_br, _ = project_perp_and_curlcurl(Abr, q0, q1, q2, q3)
                N2r = (Abr * Abr).flatten(1).sum(-1)
                C1r = (Abr * curlbr).flatten(1).sum(-1)
                C2r = (curlbr * curlbr).flatten(1).sum(-1)
                N4r = (Abr ** 4).flatten(1).sum(-1)

                N2_br.append(N2r.cpu().numpy())
                C1_br.append(C1r.cpu().numpy())
                C2_br.append(C2r.cpu().numpy())
                N4_br.append(N4r.cpu().numpy())

                Apr = Aperp_br.reshape(B, d, -1)
                Pr = (Apr.real * Apr.real + Apr.imag * Apr.imag).mean(dim=(0, 1))
                Pk_br += Pr[lowk_idx_c].cpu().numpy()

                nsamp += 1

        if (t + 1) % max(200, args.thin * 10) == 0:
            print(f"[step {t+1}/{args.steps}] nsamp={nsamp}  wall={time.time()-t0:.1f}s")

    # stack
    def cat(xs):
        return np.concatenate(xs, axis=0) if xs else np.empty((0,), dtype=np.float64)

    N2_f = cat(N2_f); C1_f = cat(C1_f); C2_f = cat(C2_f); N4_f = cat(N4_f)
    N2_bn = cat(N2_bn); C1_bn = cat(C1_bn); C2_bn = cat(C2_bn); N4_bn = cat(N4_bn)
    N2_br = cat(N2_br); C1_br = cat(C1_br); C2_br = cat(C2_br); N4_br = cat(N4_br)

    if nsamp > 0:
        Pk_f /= nsamp
        Pk_bn /= nsamp
        Pk_br /= nsamp

    fit_f = fit_invprop_p2_p4(Pk_f, p2_flat, lowk_idx)
    fit_bn = fit_invprop_p2_p4(Pk_bn, p2_flat_c, lowk_idx_c)
    fit_br = fit_invprop_p2_p4(Pk_br, p2_flat_c, lowk_idx_c)

    print("\n================= P2+P4 FIT ================")
    print(f"[fine]      m_eff2={fit_f['m_eff2']:.6g}  Z={fit_f['Z']:.6g}  gamma={fit_f['gamma']:.6g}  n={fit_f['n']}")
    print(f"[block naive]m_eff2={fit_bn['m_eff2']:.6g}  Z={fit_bn['Z']:.6g}  gamma={fit_bn['gamma']:.6g}  n={fit_bn['n']}")
    print(f"[block RG]   m_eff2={fit_br['m_eff2']:.6g}  Z={fit_br['Z']:.6g}  gamma={fit_br['gamma']:.6g}  n={fit_br['n']}")
    print("============================================\n")

    np.savez(
        args.out,
        L=L, Lc=Lc, d=d, batch=B, nsamp=nsamp,
        m2=args.m2, alpha=args.alpha, lam4=args.lam4, dt=args.dt, steps=args.steps, burnin=args.burnin, thin=args.thin,
        rg_field_scale=args.rg_field_scale,
        # lowk
        lowk_idx=lowk_idx, p2_flat=p2_flat, Pk_mean=Pk_f,
        lowk_idx_c=lowk_idx_c, p2_flat_c=p2_flat_c, Pk_mean_bn=Pk_bn, Pk_mean_br=Pk_br,
        # p2+p4 fits
        m_eff2_f=fit_f["m_eff2"], Z_f=fit_f["Z"], gamma_f=fit_f["gamma"],
        m_eff2_bn=fit_bn["m_eff2"], Z_bn=fit_bn["Z"], gamma_bn=fit_bn["gamma"],
        m_eff2_br=fit_br["m_eff2"], Z_br=fit_br["Z"], gamma_br=fit_br["gamma"],
        # operator stats
        N2_f=N2_f, C1_f=C1_f, C2_f=C2_f, N4_f=N4_f,
        N2_bn=N2_bn, C1_bn=C1_bn, C2_bn=C2_bn, N4_bn=N4_bn,
        N2_br=N2_br, C1_br=C1_br, C2_br=C2_br, N4_br=N4_br,
    )
    print(f"[saved] {args.out}")


if __name__ == "__main__":
    main()


[step 200/6000] nsamp=0  wall=2.3s
[step 400/6000] nsamp=0  wall=4.5s
[step 600/6000] nsamp=0  wall=6.7s
[step 800/6000] nsamp=0  wall=8.9s
[step 1000/6000] nsamp=0  wall=11.1s
[step 1200/6000] nsamp=0  wall=13.3s
[step 1400/6000] nsamp=0  wall=15.5s
[step 1600/6000] nsamp=5  wall=17.8s
[step 1800/6000] nsamp=15  wall=20.2s
[step 2000/6000] nsamp=25  wall=22.5s
[step 2200/6000] nsamp=35  wall=24.9s
[step 2400/6000] nsamp=45  wall=27.3s
[step 2600/6000] nsamp=55  wall=29.6s
[step 2800/6000] nsamp=65  wall=32.0s
[step 3000/6000] nsamp=75  wall=34.4s
[step 3200/6000] nsamp=85  wall=36.7s
[step 3400/6000] nsamp=95  wall=39.1s
[step 3600/6000] nsamp=105  wall=41.5s
[step 3800/6000] nsamp=115  wall=43.8s
[step 4000/6000] nsamp=125  wall=46.2s
[step 4200/6000] nsamp=135  wall=48.6s
[step 4400/6000] nsamp=145  wall=50.9s
[step 4600/6000] nsamp=155  wall=53.3s
[step 4800/6000] nsamp=165  wall=55.7s
[step 5000/6000] nsamp=175  wall=58.0s
[step 5200/6000] nsamp=185  wall=60.4s
[step 5400/6000] ns

In [18]:
import numpy as np

def fit_p2_p4(Pk, p2_flat, idx, K):
    use = idx[:K]
    G = Pk[:K].astype(np.float64)
    p2 = p2_flat[use].astype(np.float64)
    p4 = p2*p2
    good = (G>0) & np.isfinite(G) & np.isfinite(p2)
    G,p2,p4 = G[good],p2[good],p4[good]
    y = 1.0/G
    X = np.stack([np.ones_like(p2), p2, p4], axis=1)
    c, *_ = np.linalg.lstsq(X, y, rcond=None)
    return c  # [m_eff2, Z, gamma]

d = np.load("unified_v3_p4_stats.npz")
idx_f = d["lowk_idx"]
p2_f  = d["p2_flat"]
Pk_f  = d["Pk_mean"]

idx_c = d["lowk_idx_c"]
p2_c  = d["p2_flat_c"]
Pk_br = d["Pk_mean_br"]

for K in [16, 32, 64, 128, 256]:
    m,Z,g = fit_p2_p4(Pk_f, p2_f, idx_f, K)
    print(f"[fine K={K:3d}] m={m:.6g} Z={Z:.6g} gamma={g:.6g}")

print()
for K in [16, 32, 64, 128, 256]:
    m,Z,g = fit_p2_p4(Pk_br, p2_c, idx_c, K)
    print(f"[blockRG K={K:3d}] m={m:.6g} Z={Z:.6g} gamma={g:.6g}")


[fine K= 16] m=1.06026 Z=1.82046 gamma=0.206746
[fine K= 32] m=0.994743 Z=3.51791 gamma=0.402635
[fine K= 64] m=0.947997 Z=5.34251 gamma=-15.4238
[fine K=128] m=1.05843 Z=3.30749 gamma=-9.23855
[fine K=256] m=1.14052 Z=1.5103 gamma=-1.04312

[blockRG K= 16] m=3.91208 Z=17.4252 gamma=1.97829
[blockRG K= 32] m=4.03981 Z=14.0839 gamma=1.5964
[blockRG K= 64] m=3.88316 Z=20.2576 gamma=-52.4732
[blockRG K=128] m=4.21419 Z=14.609 gamma=-40.2257
[blockRG K=256] m=4.65554 Z=4.97629 gamma=3.85153


In [21]:
import numpy as np

def ols_fit(G, p2):
    p4 = p2*p2
    y = 1.0/G
    X = np.stack([np.ones_like(p2), p2, p4], axis=1)
    c, *_ = np.linalg.lstsq(X, y, rcond=None)
    return c  # m,Z,gamma

d = np.load("unified_v3_p4_stats.npz", allow_pickle=False)

p2_flat = d["p2_flat_c"].astype(np.float64)
idx = d["lowk_idx_c"].astype(np.int64)
Gbr = d["Pk_mean_br"].astype(np.float64)  # already aligned to idx order in your v3 run

# p2 values in the same order as Gbr
p2_ordered = p2_flat[idx]

for K in [16, 32, 64, 128, 256]:
    G = Gbr[:K]
    p2 = p2_ordered[:K]
    c = ols_fit(G, p2)
    print(f"K={K:3d}  m={c[0]: .6g}  Z={c[1]: .6g}  gamma={c[2]: .6g}   p2_range=[{p2.min():.6g},{p2.max():.6g}]")


K= 16  m= 3.91208  Z= 17.4252  gamma= 1.97829   p2_range=[0.0380602,0.0761205]
K= 32  m= 4.03981  Z= 14.0839  gamma= 1.5964   p2_range=[0.0380602,0.0761205]
K= 64  m= 3.88316  Z= 20.2576  gamma=-52.4732   p2_range=[0.0380602,0.114181]
K=128  m= 4.21419  Z= 14.609  gamma=-40.2257   p2_range=[0.0380602,0.184507]
K=256  m= 4.65554  Z= 4.97629  gamma= 3.85153   p2_range=[0.0380602,0.260627]


In [1]:
import math
import time
import argparse
import numpy as np
import torch
import torch.fft as fft


# -----------------------------
# Fit helpers (shell-average)
# -----------------------------
def shell_average(p2, G, tol=1e-12):
    # group by p2 value; use rounding to stabilize float equality
    key = np.round(p2 / tol).astype(np.int64)
    order = np.argsort(key)
    key = key[order]
    p2s = p2[order]
    Gs = G[order]

    shell_p2 = []
    shell_G = []
    shell_w = []

    i = 0
    n = len(key)
    while i < n:
        j = i + 1
        while j < n and key[j] == key[i]:
            j += 1
        shell_p2.append(float(p2s[i]))
        shell_G.append(float(Gs[i:j].mean()))
        shell_w.append(float(j - i))
        i = j

    return np.array(shell_p2, dtype=np.float64), np.array(shell_G, dtype=np.float64), np.array(shell_w, dtype=np.float64)


def fit_p2_p4_shells(p2_shell, G_shell, w_shell, ridge=1e-10):
    # Fit 1/G = a + b p2 + c p4 on shell-averaged points with multiplicity weights.
    y = 1.0 / G_shell
    p4 = p2_shell * p2_shell
    X = np.stack([np.ones_like(p2_shell), p2_shell, p4], axis=1)

    sw = np.sqrt(w_shell)
    Xw = X * sw[:, None]
    yw = y * sw

    A = Xw.T @ Xw + ridge * np.eye(3, dtype=np.float64)
    b = Xw.T @ yw
    c = np.linalg.solve(A, b)
    cond = np.linalg.cond(X)
    return c, cond  # [m,Z,gamma], cond


# -----------------------------
# FFT lattice operators
# -----------------------------
def make_phat_1d(L: int, a: float, device: str, dtype: torch.dtype):
    f = torch.fft.fftfreq(L, d=1.0, device=device, dtype=dtype)
    return (2.0 / a) * torch.sin(math.pi * f)


def make_pgrids(ph: torch.Tensor):
    L = ph.numel()
    p0 = ph.view(L, 1, 1, 1)
    p1 = ph.view(1, L, 1, 1)
    p2 = ph.view(1, 1, L, 1)
    p3 = ph.view(1, 1, 1, L)
    return p0, p1, p2, p3


def project_perp_and_cc(A: torch.Tensor, p0, p1, p2, p3, eps=1e-12):
    spatial = (-4, -3, -2, -1)
    Ahat = fft.fftn(A, dim=spatial, norm="ortho")  # complex
    p2grid = (p0 * p0 + p1 * p1 + p2 * p2 + p3 * p3)

    dot = (p0 * Ahat[:, 0] + p1 * Ahat[:, 1] + p2 * Ahat[:, 2] + p3 * Ahat[:, 3])

    inv = torch.zeros_like(p2grid)
    mask = p2grid > 0
    inv[mask] = 1.0 / (p2grid[mask] + eps)

    factor = dot * inv
    Aperp = torch.stack(
        [
            Ahat[:, 0] - p0 * factor,
            Ahat[:, 1] - p1 * factor,
            Ahat[:, 2] - p2 * factor,
            Ahat[:, 3] - p3 * factor,
        ],
        dim=1,
    )

    cc_hat = Aperp * p2grid
    ccA = fft.ifftn(cc_hat, dim=spatial, norm="ortho").real
    return ccA, Aperp, p2grid


def pick_lowk_from_p2grid(p2grid, K):
    p2_flat = p2grid.reshape(-1).detach().cpu().numpy()
    mask = p2_flat > 0
    idx_all = np.nonzero(mask)[0]
    idx_sort = idx_all[np.argsort(p2_flat[idx_all])]
    idx = idx_sort[: int(K)].astype(np.int64)
    return p2_flat, idx


def relerr(x, targ):
    x = np.asarray(x, dtype=np.float64)
    targ = np.asarray(targ, dtype=np.float64)
    return np.abs(x - targ) / (np.abs(targ) + 1e-12)


def loss(x, targ):
    r = relerr(x, targ)
    return float(np.sqrt(np.mean(r * r)))


def damped_step(J, f, reg):
    JTJ = J.T @ J
    A = JTJ + float(reg) * np.eye(JTJ.shape[0], dtype=np.float64)
    b = J.T @ f
    return np.linalg.solve(A, b)


# -----------------------------
# CRN evaluator
# -----------------------------
@torch.no_grad()
def eval_eff_crn(
    Lc, batch, m2, alpha, beta, lam4,
    dt, steps, burnin, thin,
    K, device, dtype, init_sigma,
    seed_init, seed_noise,
    ridge_fit=1e-10
):
    d = 4
    a = 2.0

    ph = make_phat_1d(Lc, a=a, device=device, dtype=dtype)
    p0, p1, p2, p3 = make_pgrids(ph)

    gen_init = torch.Generator(device=device).manual_seed(int(seed_init))
    A = init_sigma * torch.randn((batch, d, Lc, Lc, Lc, Lc), device=device, dtype=dtype, generator=gen_init)

    # low-k indices (ascending p2)
    _, _, p2grid = project_perp_and_cc(A[:1], p0, p1, p2, p3)
    p2_flat, lowk_idx = pick_lowk_from_p2grid(p2grid, K)
    p2_order = p2_flat[lowk_idx]  # p2 aligned to Pk_mean order

    gen_noise = torch.Generator(device=device).manual_seed(int(seed_noise))
    sqrt_2dt = math.sqrt(2.0 * dt)

    Pk_acc = np.zeros((len(lowk_idx),), dtype=np.float64)
    nsamp = 0

    for t in range(steps):
        ccA, _, _ = project_perp_and_cc(A, p0, p1, p2, p3)
        cc2A, _, _ = project_perp_and_cc(ccA, p0, p1, p2, p3)

        grad = (m2 * A) + (alpha * ccA) + (beta * cc2A) + (lam4 * (A ** 3))
        noise = torch.randn(A.shape, device=device, dtype=dtype, generator=gen_noise)
        A = A - dt * grad + sqrt_2dt * noise

        if t >= burnin and ((t - burnin) % thin == 0):
            _, Aperp_hat, _ = project_perp_and_cc(A, p0, p1, p2, p3)
            Ap = Aperp_hat.reshape(batch, d, -1)
            P = (Ap.real * Ap.real + Ap.imag * Ap.imag).mean(dim=(0, 1))
            Pk_acc += P.reshape(-1)[lowk_idx].detach().cpu().numpy()
            nsamp += 1

    if nsamp == 0:
        return np.array([np.nan, np.nan, np.nan], dtype=np.float64), np.nan, 0

    Pk_mean = Pk_acc / nsamp

    # shell-average fit
    p2_shell, G_shell, w_shell = shell_average(p2_order.astype(np.float64), Pk_mean.astype(np.float64))
    if len(p2_shell) < 6:
        # refuse: not enough distinct shells to identify p^4 stably
        return np.array([np.nan, np.nan, np.nan], dtype=np.float64), np.nan, int(len(p2_shell))

    c, cond = fit_p2_p4_shells(p2_shell, G_shell, w_shell, ridge=ridge_fit)
    return c, float(cond), int(len(p2_shell))


# -----------------------------
# Main matcher
# -----------------------------
def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--inp", type=str, default="unified_v3_p4_stats.npz")

    p.add_argument("--device", type=str, default="cuda")
    p.add_argument("--dtype", type=str, default="float32", choices=["float32", "float64"])

    # matching target uses K=256 by default (your own table says this is the only sane one)
    p.add_argument("--K", type=int, default=256)

    # eval controls
    p.add_argument("--steps", type=int, default=4000)   # cheaper inner loop
    p.add_argument("--burnin", type=int, default=1200)
    p.add_argument("--thin", type=int, default=20)
    p.add_argument("--dt", type=float, default=5e-4)
    p.add_argument("--batch", type=int, default=16)
    p.add_argument("--init_sigma", type=float, default=0.20)

    # final verification controls (longer)
    p.add_argument("--steps_final", type=int, default=8000)
    p.add_argument("--burnin_final", type=int, default=2000)
    p.add_argument("--thin_final", type=int, default=20)

    # match loop
    p.add_argument("--iters", type=int, default=8)
    p.add_argument("--tol", type=float, default=0.02)
    p.add_argument("--damping", type=float, default=1.0)
    p.add_argument("--reg_step", type=float, default=1e-2)
    p.add_argument("--du", type=float, default=0.06)   # log-steps
    p.add_argument("--max_backtrack", type=int, default=4)
    p.add_argument("--ridge_fit", type=float, default=1e-10)

    # lam4 fixed from file unless overridden
    p.add_argument("--lam4", type=float, default=np.nan)

    # init guesses (defaults from target)
    p.add_argument("--m2_init", type=float, default=np.nan)
    p.add_argument("--alpha_init", type=float, default=np.nan)
    p.add_argument("--beta_init", type=float, default=np.nan)

    # CRN seeds
    p.add_argument("--seed_init", type=int, default=12345)
    p.add_argument("--seed_noise", type=int, default=67890)

    p.add_argument("--out", type=str, default="rg_match_p4_shell_crn.npz")

    args, _ = p.parse_known_args()
    return args


def main():
    args = parse_args()

    if args.device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA requested but not available")

    device = args.device
    dtype = torch.float32 if args.dtype == "float32" else torch.float64

    dat = np.load(args.inp, allow_pickle=False)
    Lc = int(dat["Lc"])
    lam4_file = float(dat["lam4"])
    lam4 = lam4_file if np.isnan(args.lam4) else float(args.lam4)

    # TARGET: compute from stored block-RG spectrum in a stable way
    p2_flat_c = dat["p2_flat_c"].astype(np.float64)
    idx_c_full = dat["lowk_idx_c"].astype(np.int64)
    Gbr_full = dat["Pk_mean_br"].astype(np.float64)
    K = int(min(args.K, len(Gbr_full), len(idx_c_full)))

    p2_order = p2_flat_c[idx_c_full[:K]]
    Gbr = Gbr_full[:K]
    p2_shell, G_shell, w_shell = shell_average(p2_order, Gbr)
    if len(p2_shell) < 6:
        raise RuntimeError(f"Target fit refused: only {len(p2_shell)} shells in K={K} (need >=6).")
    targ, condT = fit_p2_p4_shells(p2_shell, G_shell, w_shell, ridge=args.ridge_fit)

    print("\n================= TARGET (shell-fit) =================")
    print(f"Lc={Lc}  K={K}  shells={len(p2_shell)}  cond(X)={condT:.2e}  lam4={lam4}")
    print(f"target (m_eff2,Z,gamma)=({targ[0]:.6g},{targ[1]:.6g},{targ[2]:.6g})")
    print("======================================================\n")

    # init guesses and log-parametrize to keep m2,alpha,beta > 0
    m2_0 = float(targ[0]) if np.isnan(args.m2_init) else float(args.m2_init)
    a_0  = float(targ[1]) if np.isnan(args.alpha_init) else float(args.alpha_init)
    b_0  = max(float(targ[2]), 1e-6) if np.isnan(args.beta_init) else float(args.beta_init)

    u = math.log(max(m2_0, 1e-6))
    v = math.log(max(a_0, 1e-6))
    w = math.log(max(b_0, 1e-6))

    hist = []

    for it in range(args.iters):
        t0 = time.time()

        m2 = math.exp(u); alpha = math.exp(v); beta = math.exp(w)

        base, condB, shellsB = eval_eff_crn(
            Lc, args.batch, m2, alpha, beta, lam4,
            args.dt, args.steps, args.burnin, args.thin,
            K, device, dtype, args.init_sigma,
            args.seed_init, args.seed_noise,
            ridge_fit=args.ridge_fit
        )
        if not np.all(np.isfinite(base)):
            print(f"[it {it}] eval failed (base nonfinite). Stop.")
            break

        f0 = base - targ
        r0 = relerr(base, targ)
        L0 = loss(base, targ)

        print(f"[it {it}] (m2,alpha,beta)=({m2:.6g},{alpha:.6g},{beta:.6g})  shells={shellsB} cond={condB:.2e}")
        print(f"        eff=({base[0]:.6g},{base[1]:.6g},{base[2]:.6g})")
        print(f"        relerr=({r0[0]*100:.2f}%,{r0[1]*100:.2f}%,{r0[2]*100:.2f}%)  loss={L0:.4g}  wall={time.time()-t0:.1f}s\n")

        hist.append([it, m2, alpha, beta, base[0], base[1], base[2], r0[0], r0[1], r0[2], L0])

        if np.all(r0 < args.tol):
            print("[stop] tolerance met.\n")
            break

        du = float(args.du); dv = float(args.du); dw = float(args.du)

        e_u, _, _ = eval_eff_crn(Lc, args.batch, math.exp(u+du), math.exp(v), math.exp(w), lam4,
                                 args.dt, args.steps, args.burnin, args.thin, K,
                                 device, dtype, args.init_sigma, args.seed_init, args.seed_noise, args.ridge_fit)
        e_v, _, _ = eval_eff_crn(Lc, args.batch, math.exp(u), math.exp(v+dv), math.exp(w), lam4,
                                 args.dt, args.steps, args.burnin, args.thin, K,
                                 device, dtype, args.init_sigma, args.seed_init, args.seed_noise, args.ridge_fit)
        e_w, _, _ = eval_eff_crn(Lc, args.batch, math.exp(u), math.exp(v), math.exp(w+dw), lam4,
                                 args.dt, args.steps, args.burnin, args.thin, K,
                                 device, dtype, args.init_sigma, args.seed_init, args.seed_noise, args.ridge_fit)

        J = np.column_stack([(e_u - base)/du, (e_v - base)/dv, (e_w - base)/dw])  # 3x3
        dx = damped_step(J, f0, args.reg_step)

        accepted = False
        step = float(args.damping)
        for _ in range(args.max_backtrack + 1):
            u_try = u - step*dx[0]
            v_try = v - step*dx[1]
            w_try = w - step*dx[2]

            trial, _, _ = eval_eff_crn(
                Lc, args.batch, math.exp(u_try), math.exp(v_try), math.exp(w_try), lam4,
                args.dt, args.steps, args.burnin, args.thin,
                K, device, dtype, args.init_sigma,
                args.seed_init, args.seed_noise,
                ridge_fit=args.ridge_fit
            )
            L_try = loss(trial, targ)
            if np.isfinite(L_try) and (L_try < L0):
                accepted = True
                u, v, w = u_try, v_try, w_try
                break
            step *= 0.5

        if not accepted:
            print("[warn] no improving step found; stop.\n")
            break

    # Final verification with fresh noise + longer run
    m2 = math.exp(u); alpha = math.exp(v); beta = math.exp(w)
    final, condF, shellsF = eval_eff_crn(
        Lc, args.batch, m2, alpha, beta, lam4,
        args.dt, args.steps_final, args.burnin_final, args.thin_final,
        K, device, dtype, args.init_sigma,
        args.seed_init + 999, args.seed_noise + 999,
        ridge_fit=args.ridge_fit
    )
    rF = relerr(final, targ)

    print("================= FINAL (fresh noise, long) =================")
    print(f"(m2,alpha,beta)=({m2:.6g},{alpha:.6g},{beta:.6g})  shells={shellsF} cond={condF:.2e}")
    print(f"eff=({final[0]:.6g},{final[1]:.6g},{final[2]:.6g})")
    print(f"target=({targ[0]:.6g},{targ[1]:.6g},{targ[2]:.6g})")
    print(f"relerr=({rF[0]*100:.2f}%,{rF[1]*100:.2f}%,{rF[2]*100:.2f}%)")
    print("=============================================================\n")

    np.savez(
        args.out,
        Lc=Lc, K=K, lam4=lam4,
        target=targ, cond_target=condT, shells_target=len(p2_shell),
        hist=np.array(hist, dtype=np.float64),
        m2_final=m2, alpha_final=alpha, beta_final=beta,
        eff_final=final, relerr_final=rF,
        steps=args.steps, burnin=args.burnin, thin=args.thin,
        steps_final=args.steps_final, burnin_final=args.burnin_final, thin_final=args.thin_final,
        dt=args.dt, batch=args.batch, init_sigma=args.init_sigma,
        seed_init=args.seed_init, seed_noise=args.seed_noise,
        reg_step=args.reg_step, ridge_fit=args.ridge_fit, damping=args.damping, tol=args.tol
    )
    print(f"[saved] {args.out}")


if __name__ == "__main__":
    main()


FileNotFoundError: [Errno 2] No such file or directory: 'unified_v3_p4_stats.npz'